<a href="https://colab.research.google.com/github/sakibmuhtadee/MICT_2000_Project/blob/development/FL_Bangla_Forensics_Unified_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 FL-Based Bangla Digital Forensic Analysis — Unified Colab Pipeline

**Federated Learning & Blockchain-Enabled NLP Framework for Digital Forensic Analysis of Bangla Social-Media Data**

**Authors:** Ashif Rabbani, Md. Sakib Muhtadee, Jannatul Ferdous

This single notebook integrates the three original notebooks into one end-to-end pipeline:

| Stage | Original notebook | What it does |
|------|-------------------|--------------|
| **1** | `Bangla_Dataset_Preperation.ipynb` | Builds `profane_pairs.csv`, `transliteration_dataset.csv`, `forensic_dataset_bn.csv` |
| **2** | `banglat5_pipeline.ipynb` | Fine-tunes BanglaT5 for **transliteration** and **profane-unmasking** |
| **3** | `Federated_Learning_Bangla_DEBUGGED.ipynb` | Federated learning (3 architectures) + proof-of-work blockchain audit + 5-class forensic classifier + evidence UI |

---
### How to run (Colab, with **or without** a GPU)

1. `Runtime → Change runtime type → GPU` is recommended but **not required**.
2. Set **`RUN_MODE`** in the first code cell:
   * `"QUICK"` — a fast, small-scale smoke test that runs end-to-end on **CPU** in a few minutes (tiny data, few epochs/rounds). Use this to verify the whole pipeline works. Results at this scale are **not** publishable — they only prove the code runs.
   * `"FULL"` — the real research configuration (GPU strongly recommended; hours on CPU).
3. `Runtime → Run all`.

### What changed vs. the original notebooks
* Merged into one program; shared BanglaT5 utilities and the text preprocessor are now defined **once**.
* Device- and mode-adaptive config, so it runs on GPU **or** CPU.
* All external downloads are wrapped in `try/except` with a **clearly-labelled synthetic fallback**, so the pipeline still runs if Kaggle/HF/Mendeley are unreachable.
* GitHub-push cells are **opt-in** (only run if you provide credentials).
* Bug fixes: swallowed `FederatedLearningSystem` class, missing `ensemble_predict`, unreachable unmask code in the UI, and the UI↔preprocessor wrapper mismatch. See the annotations in each cell.

> **Research integrity:** the FedAvg algorithm, blockchain layer, model architectures, preprocessing pipeline and metrics are preserved exactly. Only infrastructure (device/scale/IO) and genuine bugs were changed. Any synthetic/quick-mode output is explicitly labelled.


In [1]:
# ============================================================
# 0 · Install dependencies (Colab-safe, idempotent)
# ============================================================
# Most of these are pre-installed on Colab; pip only fetches what is missing.
import importlib, subprocess, sys

def _ensure(pip_name, import_name=None):
    """Install a package only if it cannot already be imported."""
    import_name = import_name or pip_name
    try:
        importlib.import_module(import_name)
        return
    except Exception:
        pass
    print(f"Installing {pip_name} ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=False)

for pip_name, import_name in [
    ("torch", "torch"),
    ("transformers", "transformers"),
    ("sentencepiece", "sentencepiece"),   # BanglaT5 tokenizer
    ("accelerate", "accelerate"),
    ("scikit-learn", "sklearn"),
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("tqdm", "tqdm"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("ipywidgets", "ipywidgets"),
    ("datasets", "datasets"),             # BanTH loader (Stage 1)
    ("kagglehub", "kagglehub"),           # BD-SHS loader (Stage 1)
    ("requests", "requests"),
]:
    _ensure(pip_name, import_name)

print("✅ Dependency check complete.")


✅ Dependency check complete.


In [2]:
# ============================================================
# 0 · Imports · RUN_MODE · safe device detection · adaptive scale
# ============================================================
# ── Standard library ─────────────────────────────────────────────────────────
import copy, gc, glob, hashlib, html, json, logging, os, random, sys
import unicodedata, warnings
from dataclasses import dataclass, field
from datetime    import datetime, timezone
from io          import BytesIO
from pathlib     import Path
from typing      import Any, Dict, List, Optional

# ── Numeric / ML ──────────────────────────────────────────────────────────────
import numpy  as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data          import Dataset, DataLoader, random_split
from torch.optim               import AdamW
from torch.optim.lr_scheduler  import CosineAnnealingLR
from tqdm import tqdm

from transformers import (
    AutoModel, AutoModelForSeq2SeqLM, AutoTokenizer,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    accuracy_score, classification_report,
    f1_score, precision_score, recall_score,
)

warnings.filterwarnings("ignore", category=UserWarning)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s - %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("FedBanglaChain")

# ── Environment detection ─────────────────────────────────────────────────────
IS_COLAB = "google.colab" in sys.modules

# ============================================================
#  RUN_MODE  —  set this!
#    "QUICK" : tiny, fast, CPU-friendly smoke test (verifies the whole pipeline)
#    "FULL"  : real research configuration (GPU strongly recommended)
# ============================================================
RUN_MODE = "QUICK"          # <-- change to "FULL" for real experiments
assert RUN_MODE in ("QUICK", "FULL")

FORCE_CPU = False           # set True to debug on CPU even when a GPU is present

def _safe_device() -> str:
    """Probe CUDA safely; fall back to CPU on any failure."""
    if FORCE_CPU or not torch.cuda.is_available():
        return "cpu"
    try:
        t = torch.zeros(1).cuda()
        torch.cuda.synchronize()
        del t
        return "cuda"
    except Exception as exc:
        warnings.warn(f"CUDA probe failed ({exc}). Falling back to CPU.")
        return "cpu"

DEVICE  = _safe_device()
HAS_GPU = DEVICE == "cuda"

_gpu_name, _gpu_gib = None, 0.0
if HAS_GPU:
    try:
        _props   = torch.cuda.get_device_properties(0)
        _gpu_name = _props.name
        _gpu_gib  = _props.total_memory / 1024 ** 3
    except Exception:
        pass

MODEL_NAME_T5 = "csebuetnlp/banglat5"

print("PyTorch      : " + torch.__version__)
print("Running on   : " + ("Google Colab" if IS_COLAB else "local/other"))
print("Device       : " + DEVICE.upper() + (f"  ({_gpu_name}, {_gpu_gib:.1f} GiB)" if HAS_GPU else ""))
print("RUN_MODE     : " + RUN_MODE)
if not HAS_GPU:
    print("\n[note] No GPU detected — the pipeline will run on CPU.")
    print("       Keep RUN_MODE='QUICK' unless you are prepared for a very long run.")


# ── Adaptive scale: derive all heavy knobs from (RUN_MODE, device, VRAM) ───────
def _make_scale(run_mode: str, has_gpu: bool, gpu_gib: float) -> dict:
    """Single source of truth for data sizes, epochs, rounds and batch sizes."""
    if run_mode == "QUICK":
        return dict(
            # dataset caps (rows) — keep everything tiny so CPU finishes fast
            forensic_cap      = 150,
            translit_cap      = 200,
            unmask_cap        = 200,
            # BanglaT5 fine-tuning
            t5_epochs         = 1,
            t5_batch_size     = 8 if has_gpu else 4,
            # federated learning
            fl_rounds         = 2,
            fl_local_epochs   = 1,
            fl_max_seq        = 64,
            fl_batch_size     = 8 if has_gpu else 4,
            fl_grad_accum     = 1,
            fl_freeze_layers  = 0,
        )
    # FULL research configuration
    if has_gpu:
        # Colab T4/L4/A100 have no display-watchdog TDR issue, so we do NOT need
        # the aggressive seq=32 / freeze=8 workaround from the RTX-5060-Ti notebook.
        big = gpu_gib >= 24
        return dict(
            forensic_cap      = None,          # use the whole corpus
            translit_cap      = None,
            unmask_cap        = None,
            t5_epochs         = 10,
            t5_batch_size     = 16 if big else 8,
            fl_rounds         = 10,
            fl_local_epochs   = 8,
            fl_max_seq        = 128 if big else 96,
            fl_batch_size     = 16 if big else 8,
            fl_grad_accum     = 1  if big else 2,
            fl_freeze_layers  = 0,
        )
    # FULL on CPU — allowed but slow; keep sizes bounded so it is merely slow, not hopeless
    return dict(
        forensic_cap      = 2000,
        translit_cap      = 2000,
        unmask_cap        = 2000,
        t5_epochs         = 3,
        t5_batch_size     = 4,
        fl_rounds         = 5,
        fl_local_epochs   = 2,
        fl_max_seq        = 64,
        fl_batch_size     = 4,
        fl_grad_accum     = 2,
        fl_freeze_layers  = 4,
    )

SCALE = _make_scale(RUN_MODE, HAS_GPU, _gpu_gib)
print("Scale profile:")
for k, v in SCALE.items():
    print(f"    {k:18s}: {v}")


PyTorch      : 2.11.0+cu128
Running on   : Google Colab
Device       : CUDA  (Tesla T4, 14.6 GiB)
RUN_MODE     : QUICK
Scale profile:
    forensic_cap      : 150
    translit_cap      : 200
    unmask_cap        : 200
    t5_epochs         : 1
    t5_batch_size     : 8
    fl_rounds         : 2
    fl_local_epochs   : 1
    fl_max_seq        : 64
    fl_batch_size     : 8
    fl_grad_accum     : 1
    fl_freeze_layers  : 0


---
# Stage 1 · Dataset Preparation

Builds the three CSVs the rest of the pipeline consumes:

* `profane_pairs.csv` — obfuscated → clean word pairs (Nirmol + ToxLex)
* `transliteration_dataset.csv` — Romanized → Unicode Bangla (BanTH)
* `forensic_dataset_bn.csv` — 5-class forensic corpus (BD-SHS + BanHate), classes: `normal, offensive, hate_speech, violence, cyberbully`

Every external download is wrapped in `try/except`. If a source is unreachable
(e.g. Kaggle needs auth, or a link changed), the cell falls back to a **small,
clearly-labelled synthetic sample** so the pipeline still runs end-to-end.
Replace the synthetic data with the real corpora for publishable results.


In [3]:
# ============================================================
# 1.0 · Visualization setup (publication-ready figures)
# ============================================================
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
import seaborn as sns

# Registry so study-level summary charts survive later variable reassignment
DATASET_STATS: Dict[str, Any] = {}

FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

sns.set_theme(style="whitegrid", context="talk")
mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 12, "axes.titleweight": "bold", "axes.titlesize": 14,
    "axes.labelsize": 12, "figure.autolayout": True,
})

# Unified 5-class forensic schema (kept consistent across all figures)
CLASS_ORDER  = ["normal", "offensive", "hate_speech", "violence", "cyberbully"]
CLASS_COLORS = {
    "normal": "#4C9F70", "offensive": "#E1B12C", "hate_speech": "#E67E22",
    "violence": "#C0392B", "cyberbully": "#8E44AD",
}

def _try_set_bangla_font():
    """Add a Bangla font as a fallback so Bangla glyphs render in figures."""
    candidates = [
        "/usr/share/fonts/truetype/lohit-bengali/Lohit-Bengali.ttf",
        "/usr/share/fonts/truetype/Bangla/Siyamrupali.ttf",
        "/usr/share/fonts/truetype/noto/NotoSansBengali-Regular.ttf",
    ]
    for p in candidates:
        if os.path.exists(p):
            try:
                font_manager.fontManager.addfont(p)
                bengali = font_manager.FontProperties(fname=p).get_name()
                mpl.rcParams["font.family"] = ["DejaVu Sans", bengali]
                print(f"Bangla fallback font added: {bengali}")
                return
            except Exception:
                pass
    print("No Bangla font found (OK — figures use English labels).")

_try_set_bangla_font()

def save_fig(fig, name):
    """Save each figure as 300-dpi PNG and vector PDF for the manuscript."""
    png, pdf = os.path.join(FIG_DIR, name + ".png"), os.path.join(FIG_DIR, name + ".pdf")
    fig.savefig(png); fig.savefig(pdf)
    print(f"Saved: {png} | {pdf}")

def _bar(ax, labels, values, colors=None, pct=False, total=None):
    """Vertical bar chart with count (and optional %) annotations."""
    values = list(values)
    bars = ax.bar(range(len(labels)), values, color=colors, edgecolor="white", linewidth=0.8)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels)
    tot = total if total is not None else (np.nansum(values) if pct else None)
    for rect, v in zip(bars, values):
        if v is None or (isinstance(v, float) and np.isnan(v)):
            continue
        txt = f"{int(v):,}"
        if pct and tot:
            txt += f"\n({100 * v / tot:.1f}%)"
        ax.annotate(txt, (rect.get_x() + rect.get_width() / 2, v),
                    ha="center", va="bottom", fontsize=10, fontweight="bold")
    return bars

# Whether to draw the (optional) Stage-1 exploratory figures.
MAKE_FIGURES = True
print("Visualization setup complete.  MAKE_FIGURES =", MAKE_FIGURES)


No Bangla font found (OK — figures use English labels).
Visualization setup complete.  MAKE_FIGURES = True


In [4]:
# ============================================================
# 1.1 · Download helpers + (optional) GitHub push
# ============================================================
import requests

def load_file(url: str):
    """Download a CSV/Excel file from a URL into a DataFrame; None on failure."""
    try:
        print("Downloading file ...")
        response = requests.get(url, allow_redirects=True, timeout=60)
        response.raise_for_status()
        content_type = response.headers.get("Content-Type", "").lower()
        if "excel" in content_type or "spreadsheet" in content_type:
            return pd.read_excel(BytesIO(response.content))
        if "csv" in content_type or "text" in content_type:
            return pd.read_csv(BytesIO(response.content))
        try:
            return pd.read_excel(BytesIO(response.content))
        except Exception:
            return pd.read_csv(BytesIO(response.content))
    except Exception as e:
        print(f"❌ Error loading file: {e}")
        return None


# ── Optional: push artefacts to GitHub ───────────────────────────────────────
# Disabled by default. To enable, set these environment variables (e.g. via
# Colab "Secrets" / os.environ) BEFORE running:
#   GITHUB_TOKEN, GITHUB_USER, REPO_NAME, EMAIL
# and set PUSH_TO_GITHUB = True.
PUSH_TO_GITHUB = False

def upload_to_github(file_path: str, commit_message: str, branch: str = "main"):
    """Push a single file to a GitHub repo. No-op unless PUSH_TO_GITHUB is True
    and the required env vars are present."""
    if not PUSH_TO_GITHUB:
        print(f"[github] push disabled — skipping {file_path}")
        return
    import subprocess
    token = os.getenv("GITHUB_TOKEN"); user = os.getenv("GITHUB_USER")
    repo  = os.getenv("REPO_NAME");    email = os.getenv("EMAIL")
    if not all([token, user, repo, email]):
        print("[github] missing GITHUB_TOKEN/GITHUB_USER/REPO_NAME/EMAIL — skipping.")
        return
    repo_url  = f"https://{token}@github.com/{user}/{repo}.git"
    repo_path = Path(repo)
    src_abs   = os.path.abspath(file_path)
    if not os.path.exists(src_abs):
        print(f"[github] source not found: {src_abs}")
        return
    original_dir = os.getcwd()
    try:
        if not repo_path.exists():
            subprocess.run(["git", "clone", repo_url], check=True)
        os.chdir(repo_path)
        subprocess.run(["git", "fetch", "--all"], check=False)
        subprocess.run(["git", "checkout", branch], check=False)
        dest = os.path.join(os.getcwd(), os.path.basename(file_path))
        if os.path.abspath(src_abs) != os.path.abspath(dest):
            subprocess.run(["cp", src_abs, dest], check=True)
        subprocess.run(["git", "config", "--global", "user.email", email], check=True)
        subprocess.run(["git", "config", "--global", "user.name", user], check=True)
        subprocess.run(["git", "add", os.path.basename(file_path)], check=True)
        if subprocess.run(["git", "diff", "--cached", "--quiet"]).returncode != 0:
            subprocess.run(["git", "commit", "-m", commit_message], check=True)
            subprocess.run(["git", "push", "origin", branch], check=True)
            print(f"[github] pushed {file_path} → {repo}/{branch}")
        else:
            print(f"[github] no changes in {file_path} — nothing to commit.")
    except Exception as e:
        print(f"[github] error: {e}")
    finally:
        os.chdir(original_dir)

print("Download helpers ready.  PUSH_TO_GITHUB =", PUSH_TO_GITHUB)


Download helpers ready.  PUSH_TO_GITHUB = False


In [5]:
# ============================================================
# 1.2 · Profane lexicon: Nirmol + ToxLex  →  profane_words
# ============================================================
import subprocess

_SYNTH_PROFANE = [  # clearly-labelled placeholders (NOT real profanity)
    "গালি", "বাজেকথা", "অপমান", "কুৎসা", "হুমকি",
    "বদমাশ", "মিথ্যা", "প্রতারক", "কুরুচি", "অশ্লীল",
]

def _load_nirmol():
    try:
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/Sigmakib2/Nirmol.git"],
            check=True, capture_output=True,
        )
        df = pd.read_csv("Nirmol/datasets/Nirmol-v1-dataset.csv", index_col=False)
        df.rename(columns={df.columns[0]: "input_data"}, inplace=True)
        return df[["input_data"]]
    except Exception as e:
        print(f"[Nirmol] download failed ({e}) — using synthetic fallback.")
        return pd.DataFrame({"input_data": _SYNTH_PROFANE})

def _load_toxlex():
    df = load_file(
        "https://data.mendeley.com/public-files/datasets/9pz8ssmc49/files/"
        "c13b54f5-30d7-4357-8c49-37bc46e510ab/file_downloaded"
    )
    if df is None:
        print("[ToxLex] download failed — using synthetic fallback.")
        return pd.DataFrame({"input_data": _SYNTH_PROFANE[::-1]})
    if "Base_bigram" in df.columns:
        df = df.rename(columns={"Base_bigram": "input_data"})
    else:
        df = df.rename(columns={df.columns[0]: "input_data"})
    return df[["input_data"]]

nirmol_data = _load_nirmol()
toxlex_data = _load_toxlex()
print(f"Nirmol shape: {nirmol_data.shape} | ToxLex shape: {toxlex_data.shape}")

profane_words = (
    pd.concat([toxlex_data, nirmol_data], ignore_index=True)
    .dropna().drop_duplicates().reset_index(drop=True)
)
print(f"✅ Merged profane word list: {profane_words.shape[0]} unique entries")
profane_words.head()


❌ Error loading file: 403 Client Error: Forbidden for url: https://data.mendeley.com/public-files/datasets/9pz8ssmc49/files/c13b54f5-30d7-4357-8c49-37bc46e510ab/file_downloaded
[ToxLex] download failed — using synthetic fallback.
Nirmol shape: (1181, 1) | ToxLex shape: (10, 1)
✅ Merged profane word list: 1190 unique entries


,input_data
0,অশ্লীল
1,কুরুচি
2,প্রতারক
3,মিথ্যা
4,বদমাশ


In [6]:
# ============================================================
# 1.3 · Generate masked profane pairs  →  profane_pairs.csv
# ============================================================
# Preserves the original obfuscation-style logic (asterisk / dash / slash).
# The unused char-level DataLoader from the original notebook was dropped —
# only the pair DataFrame is consumed downstream (by BanglaT5 Task B).

def generate_masked_pairs(profane_words_df, column_name="input_data"):
    def mask_word(w: str):
        styles = []
        if len(w) > 2:
            pos = random.randint(1, len(w) - 2)
            styles.append(w[:pos] + "*" + w[pos + 1:])
        styles.append("-".join(list(w)))
        styles.append("/".join(list(w)))
        n_mask = random.randint(1, max(1, len(w) // 2))
        tmp = list(w)
        for p in random.sample(range(len(w)), min(n_mask, len(w))):
            tmp[p] = "*"
        styles.append("".join(tmp))
        return styles

    if column_name not in profane_words_df.columns:
        print(f"Error: column '{column_name}' not found.")
        return pd.DataFrame(columns=["input_data", "target_data"])

    pairs = []
    for idx, w in enumerate(profane_words_df[column_name]):
        if not isinstance(w, str) or len(w.strip()) == 0:
            continue
        w = w.strip()
        for m in mask_word(w):
            pairs.append((m, w))

    if not pairs:
        print("ERROR: no training pairs generated.")
        return pd.DataFrame(columns=["input_data", "target_data"])

    print(f"Training pairs generated: {len(pairs)}")
    return pd.DataFrame(pairs, columns=["input_data", "target_data"])

random.seed(42)
profane_pairs = generate_masked_pairs(profane_words, column_name="input_data")
profane_pairs.to_csv("profane_pairs.csv", index=False, encoding="utf-8-sig")
print(f"✅ Saved {len(profane_pairs)} pairs → profane_pairs.csv")
profane_pairs.head()


Training pairs generated: 4751
✅ Saved 4751 pairs → profane_pairs.csv


,input_data,target_data
0,অ*্লীল,অশ্লীল
1,অ-শ-্-ল-ী-ল,অশ্লীল
2,অ/শ/্/ল/ী/ল,অশ্লীল
3,অশ্লী*,অশ্লীল
4,কুর*চি,কুরুচি


In [7]:
# ============================================================
# 1.4 · Transliteration corpus: BanTH  →  transliteration_dataset.csv
# ============================================================
_SYNTH_TRANSLIT = [
    ("Ami tomake bhalobasi",        "আমি তোমাকে ভালোবাসি"),
    ("Kemon acho",                  "কেমন আছো"),
    ("Valo achi dhonnobad",         "ভালো আছি ধন্যবাদ"),
    ("Tumi kothay",                 "তুমি কোথায়"),
    ("Aaj amar mon valo",           "আজ আমার মন ভালো"),
    ("Se khub bhalo manush",        "সে খুব ভালো মানুষ"),
    ("Amra bsingle ache",           "আমরা একসাথে আছি"),
    ("Bangla amar bhasha",          "বাংলা আমার ভাষা"),
]

def _load_banth():
    try:
        from datasets import load_dataset
        print("Loading BanTH dataset from Hugging Face ...")
        ds = load_dataset("aplycaebous/BanTH")
        tr = ds["train"].to_pandas().rename(columns={"Text": "input_data", "bangla": "target_data"})
        te = ds["test"].to_pandas().rename(columns={"Text": "input_data", "bangla": "target_data"})
        tr = tr[["input_data", "target_data"]]
        te = te[["input_data", "target_data"]]
        return tr, te
    except Exception as e:
        print(f"[BanTH] load failed ({e}) — using synthetic fallback.")
        base = pd.DataFrame(_SYNTH_TRANSLIT, columns=["input_data", "target_data"])
        base = pd.concat([base] * 8, ignore_index=True)          # inflate a little
        cut  = int(len(base) * 0.85)
        return base.iloc[:cut].reset_index(drop=True), base.iloc[cut:].reset_index(drop=True)

train_df, test_df = _load_banth()
translit_df = pd.concat([train_df, test_df], ignore_index=True)

# Optional cap for QUICK/CPU runs
if SCALE["translit_cap"] and len(translit_df) > SCALE["translit_cap"]:
    translit_df = translit_df.sample(SCALE["translit_cap"], random_state=42).reset_index(drop=True)

train_df.to_csv("banth_train.csv", index=False)
test_df.to_csv("banth_test.csv", index=False)
translit_df.to_csv("transliteration_dataset.csv", index=False)
print(f"✅ transliteration_dataset.csv: {translit_df.shape}  (train={len(train_df)}, test={len(test_df)})")
translit_df.head(3)


Loading BanTH dataset from Hugging Face ...


README.md:   0%|          | 0.00/1.82k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/8.23M [00:00<?, ?B/s]

val.csv:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/1.02M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/29879 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3736 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3735 [00:00<?, ? examples/s]

✅ transliteration_dataset.csv: (200, 2)  (train=29879, test=3735)


,input_data,target_data
0,Bhai valo politics sikhso,ভাই ভালো পলিটিক্স শিখসো
1,Bhai Ekta ajaira proshno korar manee ki ? ...k...,ভাই একটা আজাইরা প্রশ্ন করার মানে কি?...কে কোন ...
2,A ke dhoroner shadhinota. ? 🙂,এ কে ধরনের স্বাধীনতা।?🙂


In [8]:
# ============================================================
# 1.5 · Forensic corpus: BD-SHS + BanHate  →  forensic_dataset_bn.csv
# 5 classes: normal, offensive, hate_speech, violence, cyberbully
# ============================================================

def _load_bdshs():
    """BD-SHS via kagglehub (needs Kaggle auth). Harmonise to 5-class schema."""
    try:
        import kagglehub
        print("Loading BD-SHS from Kaggle ...")
        path = kagglehub.dataset_download("naurosromim/bdshs")
        csvs = [os.path.join(r, f)
                for r, _, fs in os.walk(path) for f in fs if f.endswith(".csv")]
        if not csvs:
            raise FileNotFoundError("No CSV in BD-SHS download.")
        frames = []
        for p in csvs:
            try:
                frames.append(pd.read_csv(p, encoding="utf-8"))
            except UnicodeDecodeError:
                frames.append(pd.read_csv(p, encoding="utf-8-sig"))
        df = max(frames, key=len).copy()
        label_map = {
            "callToViolence": "violence", "callToViolence_gender": "violence",
            "callToViolence_religion_slander": "violence",
            "gender": "hate_speech", "gender_religion_slander": "hate_speech",
            "religion_slander": "hate_speech", "slander": "offensive",
        }
        df["type"] = df["type"].map(label_map).fillna("normal")
        df = df.rename(columns={"sentence": "input_data", "type": "target_data"})
        return df[["input_data", "target_data"]]
    except Exception as e:
        print(f"[BD-SHS] load failed ({e}) — using synthetic fallback.")
        return _synth_forensic(seed=1)

def _load_banhate():
    """BanHate via HuggingFace JSON. Harmonise to 5-class schema."""
    try:
        import subprocess
        print("Loading BanHate from Hugging Face ...")
        subprocess.run(
            ["wget", "-q", "-O", "Dataset.json",
             "https://huggingface.co/datasets/aplycaebous/BanHate/resolve/main/Dataset.json"],
            check=True,
        )
        df = pd.read_json("Dataset.json")
        label_map = {
            "violence":    ["abusive/violence"],
            "hate_speech": ["religious", "gender", "origin", "political"],
            "cyberbully":  ["personal offence", "body shaming"],
        }
        def _map(label_str):
            if pd.isna(label_str):
                return "normal"
            t = str(label_str).lower()
            if any(w in t for w in label_map["violence"]):    return "violence"
            if any(w in t for w in label_map["hate_speech"]): return "hate_speech"
            if any(w in t for w in label_map["cyberbully"]):  return "cyberbully"
            return "offensive"
        df = df.rename(columns={"Comment": "input_data"})
        df["target_data"] = df["Hate Category"].apply(_map)
        return df[["input_data", "target_data"]]
    except Exception as e:
        print(f"[BanHate] load failed ({e}) — using synthetic fallback.")
        return _synth_forensic(seed=2)

def _synth_forensic(seed=0, per_class=40):
    """Clearly-labelled synthetic 5-class corpus so the pipeline can run offline."""
    rng = random.Random(seed)
    templates = {
        "normal":      ["আজ আবহাওয়া খুব সুন্দর", "আমি ভালো আছি ধন্যবাদ", "চলো একসাথে খেলি"],
        "offensive":   ["তুই একটা বাজে মানুষ", "তোর কথা শুনতে ভালো লাগে না"],
        "hate_speech": ["ওরা সব খারাপ জাত", "এই গোষ্ঠীকে বিশ্বাস করা যায় না"],
        "violence":    ["ওকে মেরে ফেলা উচিত", "সবাইকে আক্রমণ করো"],
        "cyberbully":  ["তুই দেখতে অসুন্দর", "তোকে কেউ পছন্দ করে না"],
    }
    rows = []
    for cls, temps in templates.items():
        for i in range(per_class):
            rows.append({"input_data": rng.choice(temps) + f" #{i}", "target_data": cls})
    rng.shuffle(rows)
    return pd.DataFrame(rows)

bd_shs   = _load_bdshs()
ban_hate = _load_banhate()
print(f"BD-SHS: {bd_shs.shape} | BanHate: {ban_hate.shape}")

combined_forensic = pd.concat([ban_hate, bd_shs], ignore_index=True)
combined_forensic = combined_forensic.dropna(subset=["input_data"])
combined_forensic = combined_forensic.drop_duplicates(subset=["input_data"])
combined_forensic = combined_forensic.sample(frac=1, random_state=42).reset_index(drop=True)

# Optional cap for QUICK/CPU runs (stratified-ish: simple head after shuffle)
if SCALE["forensic_cap"] and len(combined_forensic) > SCALE["forensic_cap"]:
    combined_forensic = combined_forensic.groupby("target_data", group_keys=False).apply(
        lambda g: g.head(max(1, SCALE["forensic_cap"] // combined_forensic["target_data"].nunique()))
    ).reset_index(drop=True)

combined_forensic.to_csv("forensic_dataset_bn.csv", index=False)
print(f"✅ forensic_dataset_bn.csv: {combined_forensic.shape}")
print("Class distribution:")
print(combined_forensic["target_data"].value_counts())
combined_forensic.head()


Loading BD-SHS from Kaggle ...


100%|██████████| 2.23M/2.23M [00:00<00:00, 3.45MB/s]

Extracting files...


Loading BanHate from Hugging Face ...
BD-SHS: (40224, 2) | BanHate: (19203, 2)
✅ forensic_dataset_bn.csv: (150, 2)
Class distribution:
target_data
cyberbully     30
hate_speech    30
normal         30
offensive      30
violence       30
Name: count, dtype: int64


/tmp/ipykernel_4199/2460203718.py:95: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  combined_forensic = combined_forensic.groupby("target_data", group_keys=False).apply(


,input_data,target_data
0,"তুই ন্যাংটা হয়ে,বেড়া,,ফালতু মহিলা",cyberbully
1,"বাংলাদেশের দ্বারা কিছু হবেনা, এটা হল একটা জঙ্গ...",cyberbully
2,কতটুকু মুর্খ হলে এদেরকে সফলতা বলে,cyberbully
3,কচি মেয়ে দেখে চেয়ারম্যানকে বিয়ে দিয়ে দেয়া...,cyberbully
4,চুদন ভিখারী কনটেন্ট ক্রিয়েটর,cyberbully


---
# Stage 2 · BanglaT5 Fine-Tuning

Two seq2seq fine-tuning tasks on `csebuetnlp/banglat5`, sharing one set of
metrics, model classes, datasets, trainers and inference wrappers:

| Task | Input | Output | Data |
|------|-------|--------|------|
| **A — Transliteration** | `Ami tomake bhalobasi` | `আমি তোমাকে ভালোবাসি` | `transliteration_dataset.csv` |
| **B — Profane unmask**  | `অক্** প*রু*ের` | `অক্ষম পুরুষের` | `profane_pairs.csv` |

Checkpoints are written to `checkpoints_translit/best_model` and
`checkpoints_unmask/best_model`, then reused by the Stage-3 text preprocessor.

Epochs, batch size and data size follow the `SCALE` profile, so this runs on
CPU in QUICK mode and on GPU in FULL mode. Set `TRAIN_BANGLAT5 = False` to skip
Stage 2 entirely (Stage 3 then falls back to basic text cleaning).


In [9]:
# ============================================================
# 2.1 · BanglaT5 shared metrics (used by both tasks)
# ============================================================
def exact_match(preds, refs):
    return sum(p.strip() == r.strip() for p, r in zip(preds, refs)) / max(len(refs), 1)

def token_accuracy(preds, refs, tokenizer):
    correct = total = 0
    for p, r in zip(preds, refs):
        p_ids = tokenizer.encode(p, add_special_tokens=False)
        r_ids = tokenizer.encode(r, add_special_tokens=False)
        L = max(len(p_ids), len(r_ids))
        p_ids += [0] * (L - len(p_ids))
        r_ids += [0] * (L - len(r_ids))
        correct += sum(a == b for a, b in zip(p_ids, r_ids))
        total   += L
    return correct / max(total, 1)

def character_accuracy(preds, refs):
    """Character-level accuracy — most meaningful for transliteration."""
    correct = total = 0
    for p, r in zip(preds, refs):
        L = max(len(p), len(r))
        p = p.ljust(L); r = r.ljust(L)
        correct += sum(a == b for a, b in zip(p, r))
        total   += L
    return correct / max(total, 1)

def masked_word_accuracy(raw_inputs, preds, refs, prefix):
    """Score only tokens that had '*' in the input — unmask task only."""
    correct = total = 0
    for inp, pred, ref in zip(raw_inputs, preds, refs):
        inp_toks  = inp.replace(prefix, "").split()
        pred_toks = pred.split()
        ref_toks  = ref.split()
        for i, tok in enumerate(inp_toks):
            if "*" in tok:
                total   += 1
                ref_w    = ref_toks[i]  if i < len(ref_toks)  else ""
                pred_w   = pred_toks[i] if i < len(pred_toks) else ""
                correct += int(pred_w.strip() == ref_w.strip())
    return correct / max(total, 1)

print("✅ BanglaT5 metrics defined.")


✅ BanglaT5 metrics defined.


In [10]:
# ============================================================
# 2.2 · BanglaT5 model classes (transliterator + unmasker)
# ============================================================
class _BanglaT5Base(nn.Module):
    """Shared thin wrapper around AutoModelForSeq2SeqLM."""

    def __init__(self, model_name_or_path=MODEL_NAME_T5, dropout_rate=0.1):
        super().__init__()
        self.backbone = AutoModelForSeq2SeqLM.from_pretrained(model_name_or_path)
        if hasattr(self.backbone.config, "dropout_rate"):
            self.backbone.config.dropout_rate = dropout_rate

    def to(self, *args, **kwargs):
        try:
            return super().to(*args, **kwargs)
        except Exception as e:
            warnings.warn(f"Device move failed ({e}), falling back to CPU.")
            return super().to("cpu")

    def forward(self, input_ids, attention_mask, labels=None):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask,
                            labels=labels, return_dict=True)
        result = {"lm_logits": out.logits}
        if out.loss is not None:
            result["loss"] = out.loss
        return result

    @torch.no_grad()
    def predict(self, input_ids, attention_mask, tokenizer,
                num_beams=4, max_new_tokens=128, **kw):
        ids = self.backbone.generate(
            input_ids=input_ids, attention_mask=attention_mask,
            num_beams=num_beams, max_new_tokens=max_new_tokens,
            early_stopping=True, **kw,
        )
        return tokenizer.batch_decode(ids, skip_special_tokens=True)


class BanglaT5Transliterator(_BanglaT5Base):
    """Romanized/mixed-script Bangla → Unicode Bangla."""

class BanglaT5Unmasker(_BanglaT5Base):
    """Obfuscated/profane Bangla → restored Bangla."""

print("✅ BanglaT5Transliterator and BanglaT5Unmasker defined.")


✅ BanglaT5Transliterator and BanglaT5Unmasker defined.


In [11]:
# ============================================================
# 2.3 · BanglaT5 datasets (transliteration + profane unmask)
# ============================================================
class TransliterationDataset(Dataset):
    """Loads transliteration_dataset.csv; prepends 'transliterate: '."""
    PREFIX = "transliterate: "

    def __init__(self, csv_path, tokenizer, src_max_len=128, tgt_max_len=128,
                 input_col="input_data", target_col="target_data"):
        df = pd.read_csv(csv_path).dropna(subset=[input_col, target_col])
        df[input_col]  = self.PREFIX + df[input_col].astype(str).str.strip()
        df[target_col] = df[target_col].astype(str).str.strip()
        self.inputs      = df[input_col].tolist()
        self.targets     = df[target_col].tolist()
        self.tokenizer   = tokenizer
        self.src_max_len = src_max_len
        self.tgt_max_len = tgt_max_len
        logger.info(f"TransliterationDataset: {len(self.inputs):,} samples")

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        src = self.tokenizer(self.inputs[idx], max_length=self.src_max_len,
                             padding="max_length", truncation=True, return_tensors="pt")
        tgt = self.tokenizer(self.targets[idx], max_length=self.tgt_max_len,
                             padding="max_length", truncation=True, return_tensors="pt")
        labels = tgt["input_ids"].squeeze(0).clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {
            "input_ids":      src["input_ids"].squeeze(0),
            "attention_mask": src["attention_mask"].squeeze(0),
            "labels":         labels,
            "raw_input":      self.inputs[idx],
            "raw_target":     self.targets[idx],
        }


class ProfaneUnmaskDataset(Dataset):
    """Loads profane_pairs.csv; handles asterisk / dash / slash obfuscation."""
    PREFIX = "unmask: "

    def __init__(self, csv_path, tokenizer, src_max_len=128, tgt_max_len=128,
                 input_col="input_data", target_col="target_data"):
        df = pd.read_csv(csv_path).dropna(subset=[input_col, target_col])
        df[input_col]  = df[input_col].astype(str).apply(self._normalise)
        df[target_col] = df[target_col].astype(str).str.strip()
        df[input_col]  = self.PREFIX + df[input_col]
        self.inputs      = df[input_col].tolist()
        self.targets     = df[target_col].tolist()
        self.tokenizer   = tokenizer
        self.src_max_len = src_max_len
        self.tgt_max_len = tgt_max_len
        logger.info(f"ProfaneUnmaskDataset: {len(self.inputs):,} samples")

    @staticmethod
    def _normalise(text):
        text = text.strip()
        n = max(len(text), 1)
        if text.count("-") / n > 0.25:
            text = text.replace("-", "")
        elif text.count("/") / n > 0.25:
            text = text.replace("/", "")
        return text.strip()

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        src = self.tokenizer(self.inputs[idx], max_length=self.src_max_len,
                             padding="max_length", truncation=True, return_tensors="pt")
        tgt = self.tokenizer(self.targets[idx], max_length=self.tgt_max_len,
                             padding="max_length", truncation=True, return_tensors="pt")
        labels = tgt["input_ids"].squeeze(0).clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {
            "input_ids":      src["input_ids"].squeeze(0),
            "attention_mask": src["attention_mask"].squeeze(0),
            "labels":         labels,
            "raw_input":      self.inputs[idx],
            "raw_target":     self.targets[idx],
        }

print("✅ TransliterationDataset and ProfaneUnmaskDataset defined.")


✅ TransliterationDataset and ProfaneUnmaskDataset defined.


In [12]:
# ============================================================
# 2.4 · BanglaT5 trainers (transliteration + unmask)
# ============================================================
class _BanglaT5Trainer:
    """Shared training loop; subclasses supply the validation metric set."""
    PREFIX = ""

    def __init__(self, model, tokenizer, train_loader, val_loader,
                 device=DEVICE, lr=3e-4, num_epochs=10,
                 save_dir="checkpoints", patience=3):
        self.model        = model.to(device)
        self.tokenizer    = tokenizer
        self.train_loader = train_loader
        self.val_loader   = val_loader
        self.device       = device
        self.num_epochs   = num_epochs
        self.save_dir     = Path(save_dir)
        self.patience     = patience
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
        self.scheduler = CosineAnnealingLR(self.optimizer, T_max=max(1, num_epochs))
        self.history   = {"train_loss": [], "val_loss": []}

    def _to(self, batch):
        return {k: v.to(self.device) if isinstance(v, torch.Tensor) else v
                for k, v in batch.items()}

    def _train_epoch(self):
        self.model.train()
        total = 0.0
        for batch in tqdm(self.train_loader, desc="  train", leave=False):
            b   = self._to(batch)
            out = self.model(b["input_ids"], b["attention_mask"], b["labels"])
            loss = out["loss"]
            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.optimizer.step()
            total += loss.item()
        return total / max(len(self.train_loader), 1)

    def _extra_metrics(self, inputs, preds, refs):
        return {}          # overridden by subclasses

    def _val_epoch(self):
        self.model.eval()
        total = 0.0
        all_inputs, all_preds, all_refs = [], [], []
        with torch.no_grad():
            for batch in tqdm(self.val_loader, desc="  val  ", leave=False):
                b = self._to(batch)
                out = self.model(b["input_ids"], b["attention_mask"], b["labels"])
                total += out["loss"].item()
                preds = self.model.predict(b["input_ids"], b["attention_mask"], self.tokenizer)
                all_preds.extend(preds)
                all_refs.extend(batch["raw_target"])
                all_inputs.extend(batch["raw_input"])
        metrics = {"val_loss": total / max(len(self.val_loader), 1),
                   "exact_match": exact_match(all_preds, all_refs),
                   "samples": list(zip(all_inputs, all_preds, all_refs))}
        metrics.update(self._extra_metrics(all_inputs, all_preds, all_refs))
        return metrics

    def train(self):
        best_val = float("inf"); waited = 0
        for epoch in range(1, self.num_epochs + 1):
            logger.info(f"  Epoch {epoch}/{self.num_epochs}")
            tr_loss = self._train_epoch()
            val = self._val_epoch()
            self.scheduler.step()
            logger.info(f"  train={tr_loss:.4f}  val={val['val_loss']:.4f}  "
                        f"EM={val['exact_match']:.3f}")
            self.history["train_loss"].append(tr_loss)
            self.history["val_loss"].append(val["val_loss"])
            if val["val_loss"] < best_val:
                best_val = val["val_loss"]; waited = 0
                self.model.backbone.save_pretrained(self.save_dir / "best_model")
                self.tokenizer.save_pretrained(self.save_dir / "best_model")
                logger.info("  ✓ Best checkpoint saved.")
            else:
                waited += 1
                if waited >= self.patience:
                    logger.info("  Early stopping."); break
        with open(self.save_dir / "history.json", "w") as f:
            json.dump(self.history, f, indent=2)
        return self.history


class TransliterationTrainer(_BanglaT5Trainer):
    def _extra_metrics(self, inputs, preds, refs):
        return {"token_acc": token_accuracy(preds, refs, self.tokenizer),
                "char_acc":  character_accuracy(preds, refs)}

class UnmaskTrainer(_BanglaT5Trainer):
    def _extra_metrics(self, inputs, preds, refs):
        return {"token_acc": token_accuracy(preds, refs, self.tokenizer),
                "masked_word_acc": masked_word_accuracy(
                    inputs, preds, refs, ProfaneUnmaskDataset.PREFIX)}

print("✅ TransliterationTrainer and UnmaskTrainer defined.")


✅ TransliterationTrainer and UnmaskTrainer defined.


In [13]:
# ============================================================
# 2.5 · BanglaT5 inference wrappers (used by the Stage-3 preprocessor)
# ============================================================
class _BanglaT5Wrapper:
    """Load a fine-tuned checkpoint once, call anywhere.

    BUGFIX (device): tensors are sent to the model's *actual* device
    (resolved from its parameters) rather than the requested `device`.
    They can diverge if a CUDA-OOM triggered the CPU fallback in `.to()`.
    """
    PREFIX      = ""
    MODEL_CLASS = None

    def __init__(self, checkpoint_dir, model_name=MODEL_NAME_T5,
                 device=None, batch_size=16, num_beams=4, max_new_tokens=128):
        self.batch_size     = batch_size
        self.num_beams      = num_beams
        self.max_new_tokens = max_new_tokens
        self.tokenizer      = AutoTokenizer.from_pretrained(model_name)
        self.model          = self.MODEL_CLASS(checkpoint_dir).to(device or DEVICE)
        self.model.eval()
        self._model_device  = next(self.model.parameters()).device
        logger.info(f"{type(self).__name__} ready on {self._model_device}")

    def _prep(self, text):        # overridden per task
        return self.PREFIX + text.strip()

    def _run(self, texts):
        prepped = [self._prep(t) for t in texts]
        results = []
        for i in range(0, len(prepped), self.batch_size):
            chunk = prepped[i:i + self.batch_size]
            enc = self.tokenizer(chunk, return_tensors="pt", padding=True,
                                 truncation=True, max_length=128)
            enc = {k: v.to(self._model_device) for k, v in enc.items()}
            results.extend(self.model.predict(
                enc["input_ids"], enc["attention_mask"], self.tokenizer,
                num_beams=self.num_beams, max_new_tokens=self.max_new_tokens))
        return results


class TransliterationWrapper(_BanglaT5Wrapper):
    PREFIX      = TransliterationDataset.PREFIX
    MODEL_CLASS = BanglaT5Transliterator

    def transliterate(self, text):
        single = isinstance(text, str)
        out = self._run([text] if single else list(text))
        return out[0] if single else out


class UnmaskWrapper(_BanglaT5Wrapper):
    PREFIX      = ProfaneUnmaskDataset.PREFIX
    MODEL_CLASS = BanglaT5Unmasker

    def _prep(self, text):
        return self.PREFIX + ProfaneUnmaskDataset._normalise(text)

    def unmask(self, text):
        single = isinstance(text, str)
        out = self._run([text] if single else list(text))
        return out[0] if single else out

print("✅ TransliterationWrapper and UnmaskWrapper defined.")


✅ TransliterationWrapper and UnmaskWrapper defined.


In [ ]:
# ============================================================
# 2.6 · Train BanglaT5 (Task A: transliteration, Task B: unmask)
# ============================================================
# Set TRAIN_BANGLAT5 = False to skip fine-tuning (Stage 3 then uses basic
# cleaning only). Scale (epochs / batch / subset) comes from the SCALE profile.
TRAIN_BANGLAT5 = True

TRANSLIT_CKPT = "checkpoints_translit/best_model"
UNMASK_CKPT   = "checkpoints_unmask/best_model"

_num_workers = 2 if HAS_GPU else 0
_pin         = HAS_GPU

def _subset_csv(csv_path, cap, out_path):
    """Write a capped copy of a CSV for QUICK/CPU runs; return the path used."""
    if not cap:
        return csv_path
    df = pd.read_csv(csv_path)
    if len(df) > cap:
        df = df.sample(cap, random_state=42).reset_index(drop=True)
        df.to_csv(out_path, index=False)
        return out_path
    return csv_path

def train_banglat5(task):
    cfg = {
        "translit": dict(csv="transliteration_dataset.csv", cap=SCALE["translit_cap"],
                         ds=TransliterationDataset, model=BanglaT5Transliterator,
                         trainer=TransliterationTrainer, save_dir="checkpoints_translit"),
        "unmask":   dict(csv="profane_pairs.csv", cap=SCALE["unmask_cap"],
                         ds=ProfaneUnmaskDataset, model=BanglaT5Unmasker,
                         trainer=UnmaskTrainer, save_dir="checkpoints_unmask"),
    }[task]

    csv_path = _subset_csv(cfg["csv"], cfg["cap"], f"_subset_{task}.csv")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_T5)
    full = cfg["ds"](csv_path, tokenizer, src_max_len=64, tgt_max_len=64)
    if len(full) < 4:
        logger.warning(f"[{task}] too few samples ({len(full)}) — skipping training.")
        return None

    n_val   = max(1, int(len(full) * 0.10))
    n_train = len(full) - n_val
    tr, va  = random_split(full, [n_train, n_val],
                           generator=torch.Generator().manual_seed(42))
    bs = SCALE["t5_batch_size"]
    tl = DataLoader(tr, batch_size=bs, shuffle=True,  num_workers=_num_workers, pin_memory=_pin)
    vl = DataLoader(va, batch_size=bs, shuffle=False, num_workers=_num_workers, pin_memory=_pin)

    model   = cfg["model"](MODEL_NAME_T5)
    trainer = cfg["trainer"](model, tokenizer, tl, vl, device=DEVICE,
                             lr=3e-4, num_epochs=SCALE["t5_epochs"],
                             save_dir=cfg["save_dir"], patience=3)
    logger.info(f"[{task}] train={n_train} val={n_val} epochs={SCALE['t5_epochs']} bs={bs}")
    hist = trainer.train()
    # free GPU between tasks
    del model, trainer
    gc.collect()
    if HAS_GPU:
        torch.cuda.empty_cache()
    return hist

if TRAIN_BANGLAT5:
    try:
        print("\n=== Task A: Transliteration ===")
        train_banglat5("translit")
        print("\n=== Task B: Profane unmasking ===")
        train_banglat5("unmask")
        print("\n✅ BanglaT5 fine-tuning complete.")
    except Exception as e:
        logger.warning(f"BanglaT5 training failed ({e}). Stage 3 will fall back "
                       f"to basic cleaning. See traceback below.")
        import traceback; traceback.print_exc()
        TRANSLIT_CKPT = UNMASK_CKPT = None
else:
    print("TRAIN_BANGLAT5 = False — skipping Stage 2. Stage 3 uses basic cleaning.")
    TRANSLIT_CKPT = UNMASK_CKPT = None



=== Task A: Transliteration ===


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.83k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B / 1.11MB            

spiece.model: downloading bytes:           |  0.00B            

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


=== Task B: Profane unmasking ===


Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

---
# Stage 3 · Federated Learning + Blockchain + Forensic Classification

* **Text preprocessor** — cleaning + optional BanglaT5 transliteration/unmasking (Stage 2 checkpoints).
* **Federated learning** — three heterogeneous architectures (HateBertBN, BanglaHateBERT, XLM-R) trained locally and combined with **per-architecture FedAvg**.
* **Blockchain audit** — every client update and global aggregation is recorded in a proof-of-work chain with SHA-256 weight fingerprints and a cross-chain Merkle root.
* **Evidence UI** — analyse new text, record predictions on-chain, verify integrity.

All heavy settings come from the `SCALE` profile, so the same code runs on CPU (QUICK) or GPU (FULL).


In [ ]:
# ============================================================
# 3.1 · Bangla text preprocessor (cleaning + optional BanglaT5)
# ============================================================
import re

_BANGLA_RE = re.compile(r"[\u0980-\u09FF]")
_LATIN_RE  = re.compile(r"[a-zA-Z]")
_OBFUSC_RE = re.compile(r"[\*\.]{2,}|[a-zA-Z\*\.]-[a-zA-Z\*\.]")

def _has_roman(text: str) -> bool:
    """Latin characters but no Bangla script."""
    return bool(_LATIN_RE.search(text)) and not bool(_BANGLA_RE.search(text))

def _has_obfuscated(text: str) -> bool:
    """Contains obfuscation markers (repeated * / . or separated chars)."""
    return bool(_OBFUSC_RE.search(text))


class BanglaTextPreprocessor:
    """Cleans and normalises Bangla social-media text, optionally applying
    BanglaT5 transliteration and profane-unmasking when checkpoints exist."""

    def __init__(self, translit_ckpt: Optional[str] = None,
                 unmask_ckpt: Optional[str] = None, device: Optional[str] = None):
        # Public attributes so the Evidence UI can discover the wrappers.
        self.translit_wrapper: Optional[TransliterationWrapper] = None
        self.unmask_wrapper:   Optional[UnmaskWrapper]          = None

        if translit_ckpt and Path(translit_ckpt).exists():
            logger.info(f"Loading TransliterationWrapper from {translit_ckpt}")
            try:
                self.translit_wrapper = TransliterationWrapper(translit_ckpt, device=device)
            except Exception as e:
                logger.warning(f"Could not load transliteration checkpoint: {e}")
        else:
            logger.warning("translit_ckpt missing — Romanized text passed through unchanged.")

        if unmask_ckpt and Path(unmask_ckpt).exists():
            logger.info(f"Loading UnmaskWrapper from {unmask_ckpt}")
            try:
                self.unmask_wrapper = UnmaskWrapper(unmask_ckpt, device=device)
            except Exception as e:
                logger.warning(f"Could not load unmask checkpoint: {e}")
        else:
            logger.warning("unmask_ckpt missing — obfuscated text passed through unchanged.")

    # backward-compatible internal aliases
    @property
    def _transliterator(self): return self.translit_wrapper
    @property
    def _unmasker(self):       return self.unmask_wrapper

    # ── cleaning helpers (all static) ────────────────────────────────────────
    @staticmethod
    def remove_urls(text):
        text = re.sub(r"http[s]?://\S+", "", text)
        return re.sub(r"www\.\S+", "", text)

    @staticmethod
    def decode_html_entities(text):
        return html.unescape(text)

    @staticmethod
    def remove_emojis(text):
        pat = re.compile(
            "[" "\U0001F600-\U0001F64F" "\U0001F300-\U0001F5FF"
            "\U0001F680-\U0001F6FF" "\U0001F1E0-\U0001F1FF"
            "\U00002500-\U00002BEF" "\U00002702-\U000027B0"
            "\U000024C2-\U0001F251" "]+", flags=re.UNICODE)
        return pat.sub("", text)

    @staticmethod
    def normalize_mentions_hashtags(text):
        text = re.sub(r"@(\w+)", r"\1", text)
        return re.sub(r"#(\w+)", r"\1", text)

    @staticmethod
    def normalize_punctuation(text):
        text = re.sub(r"!{2,}", "!", text)
        text = re.sub(r"\?{2,}", "?", text)
        text = re.sub(r"\.{3,}", "...", text)
        return re.sub(r"।{2,}", "।", text)

    @staticmethod
    def normalize_whitespace(text):
        return re.sub(r"\s+", " ", text).strip()

    @staticmethod
    def remove_zero_width_chars(text):
        for ch in ["\u200b", "\u200c", "\u200d", "\ufeff", "\u00ad"]:
            text = text.replace(ch, "")
        return text

    # ── main pipeline ────────────────────────────────────────────────────────
    def preprocess(self, text: str) -> str:
        if not text or not isinstance(text, str):
            return ""
        text = self.decode_html_entities(text)
        text = self.remove_urls(text)
        text = self.remove_emojis(text)
        text = self.remove_zero_width_chars(text)
        text = self.normalize_mentions_hashtags(text)
        text = self.normalize_punctuation(text)
        if self.unmask_wrapper and _has_obfuscated(text):
            try:
                text = self.unmask_wrapper.unmask(text)
            except Exception as e:
                logger.warning(f"unmask failed on one item: {e}")
        if self.translit_wrapper and _has_roman(text):
            try:
                text = self.translit_wrapper.transliterate(text)
            except Exception as e:
                logger.warning(f"transliterate failed on one item: {e}")
        return self.normalize_whitespace(text)

    def preprocess_batch(self, texts: List[str]) -> List[str]:
        result = [self.preprocess(t) for t in texts]
        logger.info(f"✅ Preprocessed {len(result)} texts.")
        return result

print("✅ BanglaTextPreprocessor defined.")


In [ ]:
# ============================================================
# 3.2 · Federated learning configuration (adaptive)
# ============================================================
@dataclass
class FederatedConfig:
    model_configs: Dict[str, str] = field(default_factory=lambda: {
        "HateBertBN":     "csebuetnlp/banglabert",
        "BanglaHateBERT": "csebuetnlp/banglabert_generator",
        "XLM-R":          "xlm-roberta-base",
    })

    num_labels:  int       = 5
    # NOTE: this ordering defines the integer encoding used everywhere in Stage 3.
    label_names: List[str] = field(default_factory=lambda: [
        "normal", "offensive", "cyberbully", "hate_speech", "violence"
    ])

    # ── these are filled from the SCALE profile below (device/mode adaptive) ──
    num_rounds:             int   = 10
    num_local_epochs:       int   = 8
    max_seq_length:         int   = 128
    batch_size:             int   = 8
    grad_accum_steps:       int   = 1
    freeze_backbone_layers: int   = 0

    fraction_clients:       float = 1.0
    use_grad_checkpointing: bool  = False
    learning_rate:          float = 2e-5
    weight_decay:           float = 0.01
    warmup_ratio:           float = 0.1
    dropout_rate:           float = 0.3

    test_split:  float = 0.2
    seed:        int   = 42

    device:         str = DEVICE
    blockchain_dir: str = "blockchains"
    checkpoint_dir: str = "checkpoints"

    translit_ckpt: Optional[str] = None
    unmask_ckpt:   Optional[str] = None


CONFIG = FederatedConfig(
    num_rounds             = SCALE["fl_rounds"],
    num_local_epochs       = SCALE["fl_local_epochs"],
    max_seq_length         = SCALE["fl_max_seq"],
    batch_size             = SCALE["fl_batch_size"],
    grad_accum_steps       = SCALE["fl_grad_accum"],
    freeze_backbone_layers = SCALE["fl_freeze_layers"],
    device                 = DEVICE,
    translit_ckpt          = TRANSLIT_CKPT,
    unmask_ckpt            = UNMASK_CKPT,
)

print("Rounds       :", CONFIG.num_rounds)
print("Local epochs :", CONFIG.num_local_epochs)
print("Max seq len  :", CONFIG.max_seq_length)
print("Batch size   :", CONFIG.batch_size, "| grad-accum:", CONFIG.grad_accum_steps)
print("Freeze layers:", CONFIG.freeze_backbone_layers)
print("Models       :", list(CONFIG.model_configs.keys()))
print("Device       :", CONFIG.device)


In [ ]:
# ============================================================
# 3.3 · Load & preprocess forensic_dataset_bn.csv  →  _ds (List[Dict])
# ============================================================
CSV_PATH  = "forensic_dataset_bn.csv"
TEXT_COL  = "input_data"
LABEL_COL = "target_data"

_raw_df = pd.read_csv(CSV_PATH)
print(f"Raw dataset shape : {_raw_df.shape} | columns: {_raw_df.columns.tolist()}")

# Instantiate the preprocessor (uses BanglaT5 checkpoints if Stage 2 produced them).
# Exposed under the plain name `preprocessor` so the Evidence UI can find it.
preprocessor = BanglaTextPreprocessor(
    translit_ckpt=CONFIG.translit_ckpt,
    unmask_ckpt=CONFIG.unmask_ckpt,
    device=CONFIG.device,
)

if TEXT_COL not in _raw_df.columns:
    raise ValueError(f"Column '{TEXT_COL}' not found. Available: {_raw_df.columns.tolist()}")

logger.info(f"Preprocessing {len(_raw_df)} rows ...")
_raw_df[TEXT_COL] = preprocessor.preprocess_batch(_raw_df[TEXT_COL].fillna("").tolist())
_raw_df = _raw_df[_raw_df[TEXT_COL].str.strip() != ""].reset_index(drop=True)
logger.info(f"After cleaning : {len(_raw_df)} rows remain.")

# ── Label encoding (string → integer index via CONFIG.label_names) ───────────
_label_to_idx = {lbl: i for i, lbl in enumerate(CONFIG.label_names)}

def _encode_label(val):
    if isinstance(val, (int, float)) and not isinstance(val, bool):
        return int(val)
    s = str(val).strip().lower()
    if s in _label_to_idx:
        return _label_to_idx[s]
    try:
        return int(s)
    except ValueError:
        logger.warning(f"Unrecognised label '{val}' -> mapped to 0 ('normal')")
        return 0

_raw_df[LABEL_COL] = _raw_df[LABEL_COL].apply(_encode_label)
_unique = sorted(_raw_df[LABEL_COL].unique().tolist())
logger.info(f"Unique label indices after encoding: {_unique}")
assert all(0 <= l < CONFIG.num_labels for l in _unique), \
    f"Label index out of range [0,{CONFIG.num_labels-1}]: {_unique}"

_ds: List[Dict] = _raw_df[[TEXT_COL, LABEL_COL]].rename(
    columns={TEXT_COL: "input_data", LABEL_COL: "target_data"}
).to_dict("records")

print(f"Dataset size (List[Dict]) : {len(_ds)}")
print(f"Label mapping             : {_label_to_idx}")
print(f"Sample record             : {_ds[0] if _ds else 'EMPTY'}")


In [ ]:
# ============================================================
# 3.4 · Blockchain layer (crypto · Block · chains · registry)
# ============================================================
def _sha256(data: str) -> str:
    return hashlib.sha256(data.encode("utf-8")).hexdigest()

def _weights_fingerprint(state_dict: Dict[str, "torch.Tensor"]) -> str:
    """Deterministic SHA-256 of a full PyTorch state-dict."""
    serialisable = {
        k: np.round(v.detach().cpu().float().numpy(), 6).tolist()
        for k, v in state_dict.items()
    }
    return _sha256(json.dumps(serialisable, sort_keys=True, ensure_ascii=False))

def _merkle_root(hashes: List[str]) -> str:
    if not hashes:
        return _sha256("empty")
    layer = list(hashes)
    while len(layer) > 1:
        if len(layer) % 2 == 1:
            layer.append(layer[-1])
        layer = [_sha256(layer[i] + layer[i + 1]) for i in range(0, len(layer), 2)]
    return layer[0]


class Block:
    DIFFICULTY: int = 2

    def __init__(self, index, block_type, data, previous_hash):
        self.index         = index
        self.block_type    = block_type
        self.timestamp     = datetime.now(timezone.utc).isoformat()
        self.data          = data
        self.previous_hash = previous_hash
        self.nonce, self.hash = self._mine()

    def _compute_hash(self, nonce):
        content = json.dumps({
            "index": self.index, "timestamp": self.timestamp,
            "block_type": self.block_type, "data": self.data,
            "previous_hash": self.previous_hash, "nonce": nonce,
        }, sort_keys=True, ensure_ascii=False, default=str)
        return _sha256(content)

    def _mine(self):
        target, nonce = "0" * self.DIFFICULTY, 0
        while True:
            h = self._compute_hash(nonce)
            if h.startswith(target):
                return nonce, h
            nonce += 1

    def to_dict(self):
        return {"index": self.index, "timestamp": self.timestamp,
                "block_type": self.block_type, "data": self.data,
                "previous_hash": self.previous_hash, "nonce": self.nonce, "hash": self.hash}

    def __repr__(self):
        return f"Block(#{self.index} | {self.block_type} | {self.hash[:14]}...)"


class _BaseChain:
    def __init__(self, chain_id, persist_dir="blockchains"):
        self.chain_id    = chain_id
        self.persist_dir = persist_dir
        self.chain: List[Block] = []
        os.makedirs(persist_dir, exist_ok=True)
        self.chain.append(Block(0, "GENESIS", {
            "chain_id": chain_id,
            "message": "Genesis - Bangla Federated Forensic System",
            "created_at": datetime.now(timezone.utc).isoformat(),
        }, "0" * 64))

    @property
    def latest_block(self):
        return self.chain[-1]

    def _append(self, block_type, data):
        block = Block(len(self.chain), block_type, data, self.latest_block.hash)
        self.chain.append(block)
        self._persist()
        return block

    def verify_chain(self):
        errors = []
        for i, block in enumerate(self.chain[1:], start=1):
            if block._compute_hash(block.nonce) != block.hash:
                errors.append(f"Block #{i}: hash mismatch (tampered data)")
            if block.previous_hash != self.chain[i - 1].hash:
                errors.append(f"Block #{i}: broken link to predecessor")
        return {"chain_id": self.chain_id, "length": len(self.chain),
                "valid": not errors, "errors": errors,
                "verified_at": datetime.now(timezone.utc).isoformat()}

    def _persist(self):
        path = os.path.join(self.persist_dir, self.chain_id + ".json")
        with open(path, "w", encoding="utf-8") as f:
            json.dump([b.to_dict() for b in self.chain], f,
                      ensure_ascii=False, indent=2, default=str)

    def summary(self):
        return {"chain_id": self.chain_id, "blocks": len(self.chain),
                "block_types": [b.block_type for b in self.chain],
                "latest_hash": self.chain[-1].hash[:20] + "..."}


class ClientChain(_BaseChain):
    def __init__(self, client_id, model_type, persist_dir="blockchains"):
        super().__init__("CLIENT_" + client_id.replace(" ", "_"), persist_dir)
        self.client_id  = client_id
        self.model_type = model_type

    def record_local_update(self, fl_round, local_epochs, train_metrics, eval_metrics,
                            state_dict=None, checkpoint_path=None):
        weight_hash = "N/A"
        if state_dict is not None:
            weight_hash = _weights_fingerprint(state_dict)
            if checkpoint_path:
                os.makedirs(os.path.dirname(checkpoint_path) or ".", exist_ok=True)
                torch.save(state_dict, checkpoint_path)
        return self._append("CLIENT_UPDATE", {
            "client_id": self.client_id, "model_type": self.model_type,
            "fl_round": fl_round, "local_epochs": local_epochs,
            "train_metrics": train_metrics,
            "eval_metrics": {k: v for k, v in eval_metrics.items() if k != "report"},
            "weight_hash": weight_hash,
            "checkpoint_path": checkpoint_path or "not_saved",
            "num_samples": train_metrics.get("num_samples", 0),
        })

    def record_evidence(self, text, prediction, confidence, model_scores,
                        analyst_notes="", source="manual_input"):
        return self._append("EVIDENCE", {
            "text": text, "prediction": prediction, "confidence": confidence,
            "model_scores": model_scores, "analyst_notes": analyst_notes,
            "source": source, "client_id": self.client_id, "model_type": self.model_type,
        })

    def get_evidence_blocks(self):
        return [b.to_dict() for b in self.chain if b.block_type == "EVIDENCE"]

    def get_update_hashes(self):
        return [b.hash for b in self.chain if b.block_type == "CLIENT_UPDATE"]


class GlobalChain(_BaseChain):
    def __init__(self, persist_dir="blockchains"):
        super().__init__("GLOBAL_SERVER", persist_dir)

    def record_aggregation(self, fl_round, client_ids, client_samples,
                           client_block_hashes, aggregated_state_dict,
                           global_metrics, checkpoint_path=None):
        total = sum(client_samples)
        fedavg_weights = {cid: round(n / total, 6) for cid, n in zip(client_ids, client_samples)} if total else {}
        global_weight_hash = _weights_fingerprint(aggregated_state_dict)
        if checkpoint_path:
            os.makedirs(os.path.dirname(checkpoint_path) or ".", exist_ok=True)
            torch.save(aggregated_state_dict, checkpoint_path)
            logger.info(f"  [GlobalChain] Weights saved -> {checkpoint_path}")
        merkle = _merkle_root(client_block_hashes)
        return self._append("GLOBAL_AGGREGATE", {
            "fl_round": fl_round, "participating_clients": client_ids,
            "client_sample_counts": dict(zip(client_ids, client_samples)),
            "fedavg_weights": fedavg_weights, "total_samples": total,
            "global_weight_hash": global_weight_hash,
            "checkpoint_path": checkpoint_path or "not_saved",
            "client_block_hashes": client_block_hashes,
            "client_merkle_root": merkle, "global_metrics": global_metrics,
        })


class BlockchainRegistry:
    def __init__(self, persist_dir="blockchains"):
        self.persist_dir  = persist_dir
        self.global_chain = GlobalChain(persist_dir)
        self.client_chains: Dict[str, ClientChain] = {}

    def register_client(self, client_id, model_type):
        chain = ClientChain(client_id, model_type, self.persist_dir)
        self.client_chains[client_id] = chain
        return chain

    def verify_all(self):
        report = {
            "verified_at": datetime.now(timezone.utc).isoformat(),
            "global_chain": self.global_chain.verify_chain(),
            "client_chains": {cid: c.verify_chain() for cid, c in self.client_chains.items()},
            "cross_chain_integrity": [],
        }
        for block in self.global_chain.chain:
            if block.block_type != "GLOBAL_AGGREGATE":
                continue
            stored = block.data["client_merkle_root"]
            recomp = _merkle_root(block.data["client_block_hashes"])
            report["cross_chain_integrity"].append({
                "fl_round": block.data["fl_round"], "global_block": block.index,
                "merkle_match": recomp == stored})
        report["fully_valid"] = (
            report["global_chain"]["valid"]
            and all(v["valid"] for v in report["client_chains"].values())
            and all(c["merkle_match"] for c in report["cross_chain_integrity"]))
        return report

    def audit_trail(self):
        lines = ["=" * 70, "  BLOCKCHAIN FORENSIC AUDIT TRAIL",
                 "  Bangla Federated Hate Speech Detection System", "=" * 70, "",
                 "[GLOBAL CHAIN]  " + self.global_chain.chain_id,
                 "  Blocks: " + str(len(self.global_chain.chain))]
        for b in self.global_chain.chain:
            if b.block_type == "GENESIS":
                lines.append(f"  #{b.index:03d} GENESIS    | {b.timestamp} | {b.hash[:16]}...")
            elif b.block_type == "GLOBAL_AGGREGATE":
                d = b.data
                lines.append(f"  #{b.index:03d} AGGREGATE  Rnd {d['fl_round']:02d}"
                             f" | g_hash={d['global_weight_hash'][:14]}..."
                             f" | ckpt={os.path.basename(d['checkpoint_path'])}"
                             f" | merkle={d['client_merkle_root'][:14]}...")
        for cid, chain in self.client_chains.items():
            lines += ["", "[CLIENT CHAIN]  " + chain.chain_id,
                      f"  Model: {chain.model_type} | Blocks: {len(chain.chain)}"]
            for b in chain.chain:
                if b.block_type == "GENESIS":
                    lines.append(f"  #{b.index:03d} GENESIS    | {b.timestamp} | {b.hash[:16]}...")
                elif b.block_type == "CLIENT_UPDATE":
                    d = b.data
                    lines.append(f"  #{b.index:03d} UPDATE     Rnd {d['fl_round']:02d}"
                                 f" | acc={d['train_metrics'].get('train_acc','?')}"
                                 f" | f1={d['eval_metrics'].get('f1_macro','?')}"
                                 f" | w={d['weight_hash'][:14]}...")
                elif b.block_type == "EVIDENCE":
                    d = b.data
                    lines.append(f"  #{b.index:03d} EVIDENCE   | pred={d['prediction']}"
                                 f" ({round(d['confidence']*100,1)}%) | src={d['source']}"
                                 f" | {d['text'][:40]}...")
        v = self.verify_all()
        lines += ["", "=" * 70, "  INTEGRITY VERIFICATION", "=" * 70,
                  "  Overall Valid : " + ("YES" if v["fully_valid"] else "NO"),
                  "  Global Chain  : " + ("VALID" if v["global_chain"]["valid"] else "INVALID")]
        for cid, res in v["client_chains"].items():
            lines.append(f"  {cid} : " + ("VALID" if res["valid"] else "INVALID")
                         + f" ({res['length']} blocks)")
        lines.append("  Cross-Chain Merkle:")
        for cc in v["cross_chain_integrity"]:
            st = "MATCH" if cc["merkle_match"] else "MISMATCH"
            lines.append(f"  Round {cc['fl_round']:02d} GlobalBlock #{cc['global_block']:03d}: {st}")
        return "\n".join(lines)

    def save_audit_report(self, path="audit_report.txt"):
        with open(path, "w", encoding="utf-8") as f:
            f.write(self.audit_trail())

    def export_all(self, path="blockchain_export.json"):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({
                "exported_at": datetime.now(timezone.utc).isoformat(),
                "global_chain": [b.to_dict() for b in self.global_chain.chain],
                "client_chains": {cid: [b.to_dict() for b in c.chain]
                                  for cid, c in self.client_chains.items()},
            }, f, ensure_ascii=False, indent=2, default=str)
        return path

print("✅ Blockchain layer defined.")


In [ ]:
# ============================================================
# 3.5 · Forensic dataset + three-architecture classifier
# ============================================================
class BanglaHateSpeechDataset(Dataset):
    """Wraps List[Dict] records: {'input_data': str, 'target_data': int|str}."""

    def __init__(self, data: List[Dict], tokenizer, max_length: int = 128):
        self.data       = data
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = item.get("input_data", "")
        if not isinstance(text, str) or not text.strip():
            text = "[PAD]"
        enc = self.tokenizer(text, max_length=self.max_length, padding="max_length",
                             truncation=True, return_tensors="pt")
        # accept both pre-encoded ints and string labels
        raw = item["target_data"]
        label = raw if isinstance(raw, int) else _label_to_idx.get(str(raw).strip().lower(), 0)
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(label, dtype=torch.long),
        }


class BanglaHateClassifier(nn.Module):
    """Three distinct fusion heads, one per client architecture.

    | Client | Backbone            | Head                             |
    |--------|---------------------|----------------------------------|
    | 1      | banglabert          | CNN(k=3,5,7)+BiLSTM+MLP fusion   |
    | 2      | banglabert_gen      | Attention pooling + MLP          |
    | 3      | xlm-roberta-base    | Masked mean pooling + MLP        |
    """

    def __init__(self, model_name, model_type, num_labels=5, dropout_rate=0.3):
        super().__init__()
        self.model_type = model_type
        self.backbone   = AutoModel.from_pretrained(model_name)
        H               = self.backbone.config.hidden_size
        self.dropout    = nn.Dropout(dropout_rate)

        if model_type == "HateBertBN":
            self.conv1  = nn.Conv1d(H, 256, kernel_size=3, padding=1)
            self.conv2  = nn.Conv1d(H, 256, kernel_size=5, padding=2)
            self.conv3  = nn.Conv1d(H, 256, kernel_size=7, padding=3)
            self.bilstm = nn.LSTM(H, 256, batch_first=True, bidirectional=True,
                                  num_layers=2, dropout=0.0)
            self.mlp    = nn.Sequential(nn.Linear(H, 512), nn.GELU(),
                                        nn.Dropout(dropout_rate), nn.Linear(512, 256))
            self.fusion = nn.Sequential(
                nn.Linear(3 * 256 + 512 + 256, 512), nn.GELU(),
                nn.Dropout(dropout_rate), nn.Linear(512, 128), nn.GELU())
            self.classifier = nn.Linear(128, num_labels)
        elif model_type == "BanglaHateBERT":
            self.attn_pool  = nn.Linear(H, 1)
            self.mlp_head   = nn.Sequential(
                nn.Linear(H, 512), nn.GELU(), nn.Dropout(dropout_rate),
                nn.Linear(512, 256), nn.GELU(), nn.Dropout(dropout_rate))
            self.classifier = nn.Linear(256, num_labels)
        elif model_type == "XLM-R":
            self.mlp_head   = nn.Sequential(
                nn.Linear(H, 512), nn.GELU(), nn.Dropout(dropout_rate),
                nn.Linear(512, 256), nn.GELU(), nn.Dropout(dropout_rate))
            self.classifier = nn.Linear(256, num_labels)
        else:
            raise ValueError(f"Unknown model_type: {model_type}")

    def to(self, *args, **kwargs):
        try:
            return super().to(*args, **kwargs)
        except Exception as exc:
            warnings.warn(f"Device move failed ({exc}). Falling back to CPU.")
            return super().to("cpu")

    def forward(self, input_ids, attention_mask):
        out    = self.backbone(input_ids=input_ids, attention_mask=attention_mask,
                               return_dict=True)
        hidden = out.last_hidden_state
        cls    = hidden[:, 0, :]

        if self.model_type == "HateBertBN":
            h_t = hidden.permute(0, 2, 1)
            c1  = F.relu(self.conv1(h_t)).max(dim=-1).values
            c2  = F.relu(self.conv2(h_t)).max(dim=-1).values
            c3  = F.relu(self.conv3(h_t)).max(dim=-1).values
            _, (hn, _) = self.bilstm(hidden)
            lstm_feat  = torch.cat([hn[-2], hn[-1]], dim=-1)
            mlp_feat   = self.mlp(self.dropout(cls))
            fused      = torch.cat([c1, c2, c3, lstm_feat, mlp_feat], dim=-1)
            return self.classifier(self.fusion(self.dropout(fused)))
        elif self.model_type == "BanglaHateBERT":
            scores  = self.attn_pool(hidden).squeeze(-1)
            scores  = scores.masked_fill(attention_mask == 0, -1e9)
            weights = torch.softmax(scores, dim=-1).unsqueeze(-1)
            pooled  = (hidden * weights).sum(dim=1)
            return self.classifier(self.mlp_head(self.dropout(pooled)))
        elif self.model_type == "XLM-R":
            mask   = attention_mask.unsqueeze(-1).float()
            pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
            return self.classifier(self.mlp_head(self.dropout(pooled)))

print("✅ BanglaHateSpeechDataset and BanglaHateClassifier defined.")


In [ ]:
# ============================================================
# 3.6 · Federated client (local training + evaluation + on-chain record)
# ============================================================
class FederatedClient:
    def __init__(self, client_id, model_name, model_type,
                 train_data, val_data, config, blockchain_registry):
        self.client_id  = client_id
        self.model_type = model_type
        self.config     = config

        self.chain = blockchain_registry.register_client(client_id, model_type)
        logger.info(f"[{client_id}] Chain = {self.chain.chain_id}")
        logger.info(f"[{client_id}] Loading {model_type} ({model_name}) ...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        # try GPU; fall back to CPU on OOM, then update self.device
        _target = torch.device(config.device)
        _model  = BanglaHateClassifier(model_name, model_type,
                                       config.num_labels, config.dropout_rate)
        try:
            _model = _model.to(_target)
            if _target.type == "cuda":
                _probe = torch.zeros(1, device=_target); del _probe
            self.device = _target
        except RuntimeError:
            warnings.warn(f"[{client_id}] GPU OOM on load — falling back to CPU.")
            _model = _model.to("cpu")
            self.device = torch.device("cpu")
        self.model = _model

        # optionally freeze first N backbone layers (cuts backward-pass time)
        _freeze_n = getattr(config, "freeze_backbone_layers", 0)
        if _freeze_n > 0 and hasattr(self.model, "backbone"):
            if hasattr(self.model.backbone, "encoder"):
                enc = self.model.backbone.encoder
                _layers = getattr(enc, "layer", None) or getattr(enc, "block", None)
                if _layers is not None:
                    for layer in list(_layers)[:_freeze_n]:
                        for p in layer.parameters():
                            p.requires_grad = False
                    frozen = sum(1 for p in self.model.backbone.parameters() if not p.requires_grad)
                    total  = sum(1 for p in self.model.backbone.parameters())
                    logger.info(f"[{client_id}] Froze first {_freeze_n} backbone layers "
                                f"({frozen}/{total} params frozen)")

        if getattr(config, "use_grad_checkpointing", False):
            if hasattr(self.model.backbone, "gradient_checkpointing_enable"):
                self.model.backbone.gradient_checkpointing_enable()
                logger.info(f"[{client_id}] Gradient checkpointing enabled.")

        _pin = str(self.device) != "cpu"
        self.train_loader = DataLoader(
            BanglaHateSpeechDataset(train_data, self.tokenizer, config.max_seq_length),
            batch_size=config.batch_size, shuffle=True, pin_memory=_pin)
        self.val_loader = DataLoader(
            BanglaHateSpeechDataset(val_data, self.tokenizer, config.max_seq_length),
            batch_size=config.batch_size, shuffle=False, pin_memory=_pin)
        self.history: List[Dict] = []

    def _to_device(self, *tensors):
        moved = [t.to(self.device) for t in tensors]
        return moved[0] if len(moved) == 1 else moved

    def local_train(self, fl_round, global_weights=None):
        # load aggregated weights on CPU first, then a single move to device
        if global_weights is not None:
            self.model.cpu()
            self.model.load_state_dict(global_weights)
            self.model.to(self.device)

        self.model.train()
        optimizer = AdamW(self.model.parameters(),
                          lr=self.config.learning_rate,
                          weight_decay=self.config.weight_decay)
        grad_accum  = max(1, getattr(self.config, "grad_accum_steps", 1))
        total_steps = max(1, len(self.train_loader) // grad_accum * self.config.num_local_epochs)
        scheduler   = get_linear_schedule_with_warmup(
            optimizer, int(total_steps * self.config.warmup_ratio), total_steps)
        criterion   = nn.CrossEntropyLoss()

        total_loss = total_correct = total_samples = 0
        for epoch in range(self.config.num_local_epochs):
            ep_loss = 0.0
            optimizer.zero_grad()
            for step, batch in enumerate(self.train_loader):
                ids, mask, lbl = self._to_device(
                    batch["input_ids"], batch["attention_mask"], batch["labels"])
                logits = self.model(ids, mask)
                loss   = criterion(logits, lbl) / grad_accum
                loss.backward()

                total_correct += (logits.detach().argmax(-1) == lbl).sum().item()
                total_samples += lbl.size(0)
                ep_loss       += loss.item() * grad_accum
                total_loss    += loss.item() * grad_accum

                if (step + 1) % grad_accum == 0 or (step + 1) == len(self.train_loader):
                    nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                    optimizer.step(); scheduler.step(); optimizer.zero_grad()

                del ids, mask, lbl, logits, loss
                if str(self.device) != "cpu":
                    torch.cuda.empty_cache()
            logger.info(f"  [{self.client_id}] Ep {epoch+1}/{self.config.num_local_epochs}"
                        f"  loss={round(ep_loss / max(len(self.train_loader),1), 4)}")

        train_metrics = {
            "train_loss": round(total_loss / max(len(self.train_loader) * self.config.num_local_epochs, 1), 4),
            "train_acc":  round(total_correct / max(total_samples, 1), 4),
            "num_samples": total_samples,
        }
        eval_metrics = self.evaluate()

        ckpt_dir  = os.path.join(self.config.checkpoint_dir,
                                 "client_" + self.client_id.replace(" ", "_"))
        ckpt_path = os.path.join(ckpt_dir, f"round_{fl_round}_{self.model_type}.pt")
        block = self.chain.record_local_update(
            fl_round, self.config.num_local_epochs, train_metrics, eval_metrics,
            self.model.state_dict(), ckpt_path)
        logger.info(f"  [{self.client_id}] Block #{block.index} | hash={block.hash[:16]}..."
                    f" | w={block.data['weight_hash'][:14]}...")
        self.history.append({**train_metrics, "fl_round": fl_round, "block_hash": block.hash})
        return train_metrics

    def evaluate(self):
        self.model.to(self.device)
        self.model.eval()
        preds_all, labels_all = [], []
        with torch.no_grad():
            for batch in self.val_loader:
                ids, mask, lbl = self._to_device(
                    batch["input_ids"], batch["attention_mask"], batch["labels"])
                preds_all.extend(self.model(ids, mask).argmax(-1).cpu().numpy())
                labels_all.extend(lbl.cpu().numpy())
                if str(self.device) != "cpu":
                    torch.cuda.empty_cache()
        self.model.cpu(); gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        if not labels_all:
            return {"accuracy": 0.0, "f1_macro": 0.0, "precision": 0.0,
                    "recall": 0.0, "report": "no validation data"}
        acc  = accuracy_score(labels_all, preds_all)
        f1   = f1_score(labels_all, preds_all, average="macro", zero_division=0)
        prec = precision_score(labels_all, preds_all, average="macro", zero_division=0)
        rec  = recall_score(labels_all, preds_all, average="macro", zero_division=0)
        return {"accuracy": round(float(acc), 4), "f1_macro": round(float(f1), 4),
                "precision": round(float(prec), 4), "recall": round(float(rec), 4),
                "report": classification_report(labels_all, preds_all,
                    target_names=self.config.label_names, zero_division=0)}

    def predict_and_record(self, text, analyst_notes="", source="manual_input"):
        self.model.to(self.device)
        self.model.eval()
        enc = self.tokenizer(text, max_length=self.config.max_seq_length,
                             padding="max_length", truncation=True, return_tensors="pt")
        with torch.no_grad():
            logits = self.model(enc["input_ids"].to(self.device),
                                enc["attention_mask"].to(self.device))
            probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
        pred_idx   = int(probs.argmax())
        prediction = self.config.label_names[pred_idx]
        confidence = float(probs[pred_idx])
        model_scores = {lbl: round(float(probs[i]), 4)
                        for i, lbl in enumerate(self.config.label_names)}
        block = self.chain.record_evidence(
            text=text, prediction=prediction, confidence=confidence,
            model_scores=model_scores, analyst_notes=analyst_notes, source=source)
        return {"text": text, "prediction": prediction, "confidence": confidence,
                "scores": model_scores, "block_hash": block.hash,
                "block_index": block.index, "model_type": self.model_type}

    def get_weights(self):
        return copy.deepcopy(self.model.state_dict())

    def get_num_samples(self):
        return len(self.train_loader.dataset)

    def get_latest_block_hash(self):
        return self.chain.latest_block.hash

print("✅ FederatedClient defined.")


In [ ]:
# ============================================================
# 3.7 · Federated server (per-architecture FedAvg on CPU)
# ============================================================
class FederatedServer:
    def __init__(self, config, blockchain_registry):
        self.config   = config
        self.registry = blockchain_registry
        self.arch_weights: Dict[str, Optional[Dict]] = {
            arch: None for arch in config.model_configs.keys()}

    def aggregate_per_arch(self, fl_round, clients, global_metrics):
        arch_groups: Dict[str, List["FederatedClient"]] = {}
        for c in clients:
            arch_groups.setdefault(c.model_type, []).append(c)

        all_cids, all_n, all_hashes = [], [], []
        new_arch_weights: Dict[str, Dict] = {}

        for arch, arch_clients in arch_groups.items():
            w_list = [c.get_weights()     for c in arch_clients]
            n_list = [c.get_num_samples() for c in arch_clients]
            total  = sum(n_list) or 1
            logger.info(f"  [Server] FedAvg {arch}: {len(arch_clients)} clients, {total} samples")

            # aggregate on CPU to avoid VRAM OOM during FedAvg
            agg = {key: torch.zeros_like(t.cpu(), dtype=torch.float32)
                   for key, t in w_list[0].items()}
            for w, n in zip(w_list, n_list):
                proportion = n / total
                for key in agg:
                    agg[key] += proportion * w[key].detach().cpu().float()

            new_arch_weights[arch]  = agg
            self.arch_weights[arch] = agg

            all_cids.extend([c.client_id for c in arch_clients])
            all_n.extend(n_list)
            all_hashes.extend([c.get_latest_block_hash() for c in arch_clients])

            ckpt_dir  = os.path.join(self.config.checkpoint_dir, "global")
            os.makedirs(ckpt_dir, exist_ok=True)
            ckpt_path = os.path.join(ckpt_dir, f"round_{fl_round}_{arch}.pt")
            torch.save(agg, ckpt_path)
            logger.info(f"  [Server] {arch} weights -> {ckpt_path}")

        primary_arch = sorted(new_arch_weights.keys())[0]
        primary_ckpt = os.path.join(self.config.checkpoint_dir, "global",
                                    f"round_{fl_round}_primary_{primary_arch}.pt")
        block = self.registry.global_chain.record_aggregation(
            fl_round=fl_round, client_ids=all_cids, client_samples=all_n,
            client_block_hashes=all_hashes,
            aggregated_state_dict=new_arch_weights[primary_arch],
            global_metrics={**global_metrics,
                "arch_groups": {a: len(cs) for a, cs in arch_groups.items()},
                "all_arch_ckpts": {a: f"round_{fl_round}_{a}.pt" for a in new_arch_weights}},
            checkpoint_path=primary_ckpt)
        logger.info(f"[Server] GlobalBlock #{block.index} | hash={block.hash[:16]}..."
                    f" | g_hash={block.data['global_weight_hash'][:14]}..."
                    f" | merkle={block.data['client_merkle_root'][:14]}...")
        return new_arch_weights

    def get_weights_for(self, model_type):
        return self.arch_weights.get(model_type)

    def select_clients(self, clients):
        k = max(1, int(len(clients) * self.config.fraction_clients))
        return random.sample(clients, k)

print("✅ FederatedServer defined.")


In [ ]:
# ============================================================
# 3.8 · Orchestrator — FederatedLearningSystem
# ============================================================
# BUGFIX: in the original notebook a stray comment swallowed a second
# `class FederatedLearningSystem:` header, and `ensemble_predict` (referenced by
# the CLI fallback) was never defined. This is one clean class with every
# method defined exactly once, including `ensemble_predict`.
class FederatedLearningSystem:
    def __init__(self, config):
        self.config    = config
        self.registry  = BlockchainRegistry(config.blockchain_dir)
        self.server    = FederatedServer(config, self.registry)
        self.clients: List[FederatedClient] = []
        self.test_data: List[Dict] = []
        random.seed(config.seed)
        np.random.seed(config.seed)
        torch.manual_seed(config.seed)

    def _iid_partition(self, data, n):
        random.shuffle(data)
        sz = max(1, len(data) // n)
        return [data[i * sz:(i + 1) * sz] for i in range(n)]

    def setup(self):
        logger.info("=" * 65)
        logger.info("  Bangla FL + Blockchain  |  per-architecture FedAvg")
        logger.info("=" * 65)
        all_data = list(_ds)
        split    = int(len(all_data) * (1 - self.config.test_split))
        train_data     = all_data[:split]
        self.test_data = all_data[split:]
        logger.info(f"Dataset  train={len(train_data)}  test={len(self.test_data)}")

        client_cfgs = list(self.config.model_configs.items())
        partitions  = self._iid_partition(train_data, len(client_cfgs))
        for i, (model_type, model_name) in enumerate(client_cfgs):
            local = partitions[i]
            v_sz  = max(1, int(len(local) * 0.2))
            client = FederatedClient(
                client_id=f"Client_{i+1}_{model_type}",
                model_name=model_name, model_type=model_type,
                train_data=local[v_sz:], val_data=local[:v_sz],
                config=self.config, blockchain_registry=self.registry)
            self.clients.append(client)
            logger.info(f"[Setup] Client {i+1} ({model_type})  "
                        f"train={len(local[v_sz:])}  val={len(local[:v_sz])}")

    @staticmethod
    def _safe_load_state(model, state_dict, device):
        """Load a CPU state_dict without holding two GPU copies at once."""
        model.cpu()
        model.load_state_dict(state_dict)
        model.to(device)

    def _train_one_round(self, rnd, all_rounds):
        logger.info("\n" + "-" * 65)
        logger.info(f"  ROUND {rnd}/{self.config.num_rounds}")
        logger.info("-" * 65)
        selected = self.server.select_clients(self.clients)

        for client in selected:
            arch_w = self.server.get_weights_for(client.model_type)
            logger.info(f"\n[R{rnd}] Training {client.client_id} ...")
            client.model.to(client.device)
            client.local_train(rnd, arch_w)
            client.model.cpu(); gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            logger.info(f"  [{client.client_id}] done — GPU cleared")

        pre_eval = [c.evaluate() for c in self.clients]
        global_metrics = {
            "avg_accuracy":  round(float(np.mean([m["accuracy"]  for m in pre_eval])), 4),
            "avg_f1_macro":  round(float(np.mean([m["f1_macro"]  for m in pre_eval])), 4),
            "avg_precision": round(float(np.mean([m["precision"] for m in pre_eval])), 4),
            "avg_recall":    round(float(np.mean([m["recall"]    for m in pre_eval])), 4),
        }
        new_arch_w = self.server.aggregate_per_arch(rnd, selected, global_metrics)

        for client in self.clients:
            aw = new_arch_w.get(client.model_type)
            if aw is not None:
                self._safe_load_state(client.model, aw, client.device)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        post_eval = [c.evaluate() for c in self.clients]
        for client, em in zip(self.clients, post_eval):
            logger.info(f"  [{client.client_id}]  acc={em['accuracy']}  f1={em['f1_macro']}"
                        f"  prec={em['precision']}  rec={em['recall']}")
        avg_acc = round(float(np.mean([m["accuracy"] for m in post_eval])), 4)
        avg_f1  = round(float(np.mean([m["f1_macro"] for m in post_eval])), 4)
        logger.info(f"\n[R{rnd}] * Avg Acc={avg_acc}  F1={avg_f1}")

        all_rounds.append({
            "round": rnd, **global_metrics,
            "arch_weights_saved": list(new_arch_w.keys()),
            "post_eval": [{k: v for k, v in m.items() if k != "report"} for m in post_eval],
        })
        with open("fedavg_results.json", "w", encoding="utf-8") as f:
            json.dump(all_rounds, f, ensure_ascii=False, indent=2)
        logger.info(f"[R{rnd}] Results saved ({len(all_rounds)} rounds total)")
        return all_rounds

    def run(self):
        all_rounds = []
        for rnd in range(1, self.config.num_rounds + 1):
            all_rounds = self._train_one_round(rnd, all_rounds)
        self._finalize(all_rounds)
        return all_rounds

    def run_from(self, start_round, prev_results=None):
        all_rounds = list(prev_results) if prev_results else []
        if not all_rounds and os.path.isfile("fedavg_results.json"):
            with open("fedavg_results.json", encoding="utf-8") as f:
                all_rounds = [r for r in json.load(f) if r["round"] <= start_round]
        for rnd in range(start_round + 1, self.config.num_rounds + 1):
            all_rounds = self._train_one_round(rnd, all_rounds)
        self._finalize(all_rounds)
        return all_rounds

    def resume_from_checkpoint(self):
        global_ckpt_dir = os.path.join(self.config.checkpoint_dir, "global")
        if not os.path.isdir(global_ckpt_dir):
            logger.info("[Resume] No checkpoint directory — starting from round 1.")
            return 0
        files = [f for f in glob.glob(os.path.join(global_ckpt_dir, "round_*.pt"))
                 if "primary_" not in os.path.basename(f)]
        if not files:
            logger.info("[Resume] No global weight files — starting from round 1.")
            return 0
        rounds_found = set()
        for f in files:
            parts = os.path.basename(f).split("_")
            try:
                rounds_found.add(int(parts[1]))
            except (IndexError, ValueError):
                pass
        if not rounds_found:
            logger.info("[Resume] Could not parse round numbers — starting from round 1.")
            return 0
        last_round = max(rounds_found)
        loaded = []
        for arch in self.config.model_configs.keys():
            ckpt_path = os.path.join(global_ckpt_dir, f"round_{last_round}_{arch}.pt")
            if not os.path.isfile(ckpt_path):
                logger.warning(f"[Resume] Missing checkpoint for {arch} at round {last_round}")
                continue
            try:
                state = torch.load(ckpt_path, map_location="cpu", weights_only=True)
            except TypeError:
                state = torch.load(ckpt_path, map_location="cpu")
            self.server.arch_weights[arch] = state
            for client in self.clients:
                if client.model_type == arch:
                    self._safe_load_state(client.model, state, client.device)
                    logger.info(f"[Resume] Loaded {arch} → {client.client_id} "
                                f"(device={client.device})")
            loaded.append(arch)
            del state; gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        logger.info(f"[Resume] Resuming from round {last_round+1}/{self.config.num_rounds} "
                    f" loaded={loaded}")
        return last_round

    @torch.no_grad()
    def ensemble_predict(self, texts):
        """Average softmax probabilities across all client architectures.
        BUGFIX: this method was referenced by the CLI fallback but never defined."""
        single = isinstance(texts, str)
        batch  = [texts] if single else list(texts)
        label_names = self.config.label_names
        results = []
        # collect per-text probability vectors from every client
        per_text_probs = [[] for _ in batch]
        for c in self.clients:
            c.model.to(c.device); c.model.eval()
            for i, t in enumerate(batch):
                enc = c.tokenizer(t, max_length=self.config.max_seq_length,
                                  padding="max_length", truncation=True, return_tensors="pt")
                logits = c.model(enc["input_ids"].to(c.device),
                                 enc["attention_mask"].to(c.device))
                per_text_probs[i].append(torch.softmax(logits, dim=-1).cpu().numpy()[0])
            c.model.cpu()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        for probs_list in per_text_probs:
            avg = np.mean(probs_list, axis=0)
            idx = int(avg.argmax())
            agreement = ("unanimous"
                         if len({int(p.argmax()) for p in probs_list}) == 1 else "split")
            results.append({
                "prediction": label_names[idx],
                "confidence": float(avg[idx]),
                "scores": {lbl: float(avg[j]) for j, lbl in enumerate(label_names)},
                "model_agreement": agreement,
            })
        return results[0] if single else results

    def _finalize(self, rounds):
        with open("fedavg_results.json", "w", encoding="utf-8") as f:
            json.dump(rounds, f, ensure_ascii=False, indent=2)
        self.registry.save_audit_report("audit_report.txt")
        self.registry.export_all("blockchain_export.json")
        v = self.registry.verify_all()
        logger.info("\n" + "=" * 65)
        logger.info("  BLOCKCHAIN INTEGRITY REPORT")
        logger.info("=" * 65)
        logger.info(f"  Overall Valid : {v['fully_valid']}")
        logger.info("  Global Chain  : " + ("VALID" if v["global_chain"]["valid"] else "INVALID")
                    + f" ({v['global_chain']['length']} blocks)")
        for cid, res in v["client_chains"].items():
            logger.info(f"  {cid} : " + ("VALID" if res["valid"] else "INVALID")
                        + f" ({res['length']} blocks)")
        for cc in v["cross_chain_integrity"]:
            st = "MATCH" if cc["merkle_match"] else "MISMATCH"
            logger.info(f"  Rnd {cc['fl_round']:02d} GlobalBlock #{cc['global_block']:03d} Merkle: {st}")
        logger.info("Saved: fedavg_results.json | audit_report.txt | blockchain_export.json")

print("✅ FederatedLearningSystem defined.")


In [ ]:
# ============================================================
# 3.9 · Run the federated pipeline (resume-aware)
# ============================================================
fl_system = FederatedLearningSystem(CONFIG)
fl_system.setup()

# resume_from_checkpoint() returns 0 when there is nothing to resume, so this
# is safe on a fresh run and picks up where it left off after a crash.
last_completed_round = fl_system.resume_from_checkpoint()

if last_completed_round >= CONFIG.num_rounds and os.path.isfile("fedavg_results.json"):
    print(f"All {CONFIG.num_rounds} rounds already complete — loading results from disk.")
    with open("fedavg_results.json", encoding="utf-8") as f:
        round_results = json.load(f)
elif last_completed_round > 0:
    print(f"Resuming from round {last_completed_round + 1} / {CONFIG.num_rounds} ...")
    round_results = fl_system.run_from(last_completed_round)
else:
    round_results = fl_system.run()

print(f"\nTraining complete!  Total rounds recorded: {len(round_results)}")


In [ ]:
# ============================================================
# 3.10 · Evaluation utilities (per-round & per-class F1)
# ============================================================
@torch.no_grad()
def _collect_preds(model, loader, device):
    model.to(device); model.eval()
    preds, labels = [], []
    for batch in loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        preds.extend(model(ids, mask).argmax(-1).cpu().numpy())
        labels.extend(batch["labels"].numpy())
        if str(device) != "cpu":
            torch.cuda.empty_cache()
    model.cpu(); gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return np.array(labels), np.array(preds)


def per_class_f1_for_round(fl_system, rnd, use_test=False):
    """Per-class F1 at a round, averaged across the three architectures."""
    cfg   = fl_system.config
    gdir  = os.path.join(cfg.checkpoint_dir, "global")
    names = list(cfg.label_names)
    idx   = list(range(len(names)))
    per_client = {}
    for c in fl_system.clients:
        ckpt = os.path.join(gdir, f"round_{rnd}_{c.model_type}.pt")
        if not os.path.isfile(ckpt):
            print(f"  [round {rnd}] missing {c.model_type} checkpoint — skipping")
            continue
        try:
            state = torch.load(ckpt, map_location="cpu", weights_only=True)
        except TypeError:
            state = torch.load(ckpt, map_location="cpu")
        fl_system._safe_load_state(c.model, state, c.device)
        del state; gc.collect()
        if use_test:
            loader = DataLoader(
                BanglaHateSpeechDataset(fl_system.test_data, c.tokenizer, cfg.max_seq_length),
                batch_size=cfg.batch_size, shuffle=False,
                pin_memory=(str(c.device) != "cpu"))
        else:
            loader = c.val_loader
        y, p = _collect_preds(c.model, loader, c.device)
        f1s  = f1_score(y, p, labels=idx, average=None, zero_division=0)
        per_client[c.client_id] = {n: round(float(v), 4) for n, v in zip(names, f1s)}
    if not per_client:
        return {}, {}
    mean = {n: round(float(np.mean([d[n] for d in per_client.values()])), 4) for n in names}
    return per_client, mean


def table7_row(mean):
    """Collapse 5 per-class F1s into the manuscript's 4 reporting columns."""
    vc_parts = [mean[k] for k in ("violence", "cyberbully") if k in mean]
    vc = round(float(np.mean(vc_parts)), 4) if vc_parts else None
    return {"normal": mean.get("normal"), "offensive": mean.get("offensive"),
            "hate_speech": mean.get("hate_speech"), "violence/cyberbully": vc}


print("Per-class F1 across rounds:")
for rnd in range(1, CONFIG.num_rounds + 1):
    pcs, mean = per_class_f1_for_round(fl_system, rnd, use_test=False)
    if not mean:
        continue
    print(f"\n=== Round {rnd} — per-class F1 (validation split) ===")
    for cid, d in pcs.items():
        print(f"  {cid:24s} {d}")
    print(f"  mean across archs        {mean}")
    print(f"  Reporting columns        {table7_row(mean)}")


In [ ]:
# ============================================================
# 3.11 · Training results summary
# ============================================================
print("Round  Acc       F1        Prec      Rec       Arch Saved")
print("-" * 75)
for r in round_results:
    print(str(r["round"]).ljust(7)
          + str(r["avg_accuracy"]).ljust(10)
          + str(r["avg_f1_macro"]).ljust(10)
          + str(r["avg_precision"]).ljust(10)
          + str(r["avg_recall"]).ljust(10)
          + str(r.get("arch_weights_saved", [])))

if round_results:
    best = max(round_results, key=lambda x: x["avg_f1_macro"])
    print(f"\nBest Round : {best['round']}  F1={best['avg_f1_macro']}")


In [ ]:
# ============================================================
# 3.12 · Evidence Detection UI  (run after training completes)
# ============================================================
# BUGFIXES vs. original:
#   * uses the preprocessor's PUBLIC wrappers (translit_wrapper/unmask_wrapper)
#     and resolves the `preprocessor` global correctly — the original looked up
#     attributes that never existed, so preprocessing silently never fired;
#   * removed the early `return` that made the unmask step dead code.
import re as _re
_LATIN_RE_UI  = _re.compile(r"[a-zA-Z]")
_BANGLA_RE_UI = _re.compile(r"[\u0980-\u09FF]")
_OBFSC_RE_UI  = _re.compile(r"[\*\.]{2,}|[a-zA-Z\*\.]-[a-zA-Z\*\.]")

_LABEL_COLOUR = {"normal": "#27ae60", "offensive": "#e67e22", "cyberbully": "#e74c3c",
                 "hate_speech": "#8e44ad", "violence": "#c0392b"}
def _colour(label): return _LABEL_COLOUR.get(label, "#2c3e50")

def _needs_translit(t): return bool(_LATIN_RE_UI.search(t)) and not bool(_BANGLA_RE_UI.search(t))
def _needs_unmask(t):   return bool(_OBFSC_RE_UI.search(t))

def _resolve_preprocessor():
    return globals().get("preprocessor", None)

def _preprocess_text(text, do_translit, do_unmask):
    """Returns (processed_text, steps). No early-return bug — both steps run."""
    steps, result = [], text
    pp = _resolve_preprocessor()
    translit_wrapper = getattr(pp, "translit_wrapper", None)
    unmask_wrapper   = getattr(pp, "unmask_wrapper", None)

    if do_unmask and _needs_unmask(result):
        if unmask_wrapper is None:
            steps.append(("Profane unmask", result, "⚠️ skipped (no unmask checkpoint)"))
        else:
            try:
                out = unmask_wrapper.unmask(result)
                if out and out.strip() != result.strip():
                    steps.append(("Profane unmask", result, out)); result = out
            except Exception as exc:
                steps.append(("Profane unmask", result, f"⚠️ skipped ({exc})"))

    if do_translit and _needs_translit(result):
        if translit_wrapper is None:
            steps.append(("Transliteration", result, "⚠️ skipped (no translit checkpoint)"))
        else:
            try:
                out = translit_wrapper.transliterate(result)
                if out and out.strip() != result.strip():
                    steps.append(("Transliteration", result, out)); result = out
            except Exception as exc:
                steps.append(("Transliteration", result, f"⚠️ skipped ({exc})"))
    return result, steps

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    _HAS_WIDGETS = True
except ImportError:
    _HAS_WIDGETS = False
    print("ipywidgets not available — using CLI fallback below.")

if _HAS_WIDGETS:
    _title = widgets.HTML(
        "<h3 style='color:#1a1a2e;margin-bottom:4px'>🔍 Bangla Digital Forensic Evidence Detection</h3>"
        "<p style='color:#555;font-size:13px'>Analyse text, record on-chain evidence, verify integrity.</p><hr/>")
    _text_input = widgets.Textarea(placeholder="Enter Bangla / Romanized / obfuscated text …",
                                   layout=widgets.Layout(width="98%", height="90px"))
    _source_dd  = widgets.Dropdown(options=["manual_input", "facebook", "twitter", "youtube",
                                            "telegram", "whatsapp", "other"],
                                   value="manual_input", description="Source:",
                                   layout=widgets.Layout(width="250px"))
    _client_dd  = widgets.Dropdown(options=[c.client_id for c in fl_system.clients] + ["Ensemble (all)"],
                                   value="Ensemble (all)", description="Model:",
                                   layout=widgets.Layout(width="320px"))
    _notes_input = widgets.Text(placeholder="Analyst notes / case ID (optional)",
                                layout=widgets.Layout(width="98%"))
    _translit_toggle = widgets.Checkbox(value=True, description="Auto-transliterate Romanized → Bangla",
                                        style={"description_width": "initial"}, layout=widgets.Layout(width="98%"))
    _unmask_toggle   = widgets.Checkbox(value=True, description="Auto-unmask obfuscated/profane words",
                                        style={"description_width": "initial"}, layout=widgets.Layout(width="98%"))
    _analyse_btn = widgets.Button(description="Analyse & Record", button_style="primary",
                                  layout=widgets.Layout(width="200px"), icon="search")
    _verify_btn  = widgets.Button(description="Verify Chains", button_style="info",
                                  layout=widgets.Layout(width="160px"), icon="check")
    _audit_btn   = widgets.Button(description="Audit Trail", button_style="warning",
                                  layout=widgets.Layout(width="160px"), icon="list")
    _export_btn  = widgets.Button(description="Export JSON", button_style="success",
                                  layout=widgets.Layout(width="160px"), icon="download")
    _output = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="10px",
                             min_height="80px", max_height="600px", overflow_y="auto"))

    def _steps_html(steps):
        if not steps:
            return ""
        rows = "".join(
            f"<tr><td style='padding:4px 8px;color:#555;font-size:12px'>{lbl}</td>"
            f"<td style='padding:4px 8px;font-size:12px;color:#888'>{before}</td>"
            f"<td style='padding:4px 8px;font-size:12px'>→</td>"
            f"<td style='padding:4px 8px;font-size:12px;color:#1a1a2e'><b>{after}</b></td></tr>"
            for lbl, before, after in steps)
        return ("<details style='margin:6px 0'><summary style='cursor:pointer;font-size:12px;"
                "color:#555'>🔧 Preprocessing steps</summary>"
                f"<table style='border-collapse:collapse;margin-top:4px;width:100%'>{rows}</table></details>")

    def _on_analyse(btn):
        import traceback
        text  = _text_input.value.strip()
        notes = _notes_input.value.strip()
        src   = _source_dd.value
        model = _client_dd.value
        if not text:
            with _output:
                clear_output(); print("⚠️  Please enter some text to analyse.")
            return
        with _output:
            clear_output(); print("⏳ Preprocessing …")
        try:
            processed, steps = _preprocess_text(text, _translit_toggle.value, _unmask_toggle.value)
            if model == "Ensemble (all)":
                res = fl_system.ensemble_predict(processed)
                # record on every client's chain
                for c in fl_system.clients:
                    c.predict_and_record(processed, analyst_notes=notes, source=src)
                agreement_tag = " | Agreement: " + res.get("model_agreement", "")
            else:
                client_obj = next(c for c in fl_system.clients if c.client_id == model)
                res = client_obj.predict_and_record(processed, analyst_notes=notes, source=src)
                agreement_tag = ""
            label, conf = res["prediction"], res["confidence"]
            col = _colour(label)
            input_block = ""
            if processed != text:
                input_block = ("<div style='background:#f0f4ff;border-radius:4px;padding:6px 10px;"
                               f"margin:4px 0;font-size:12px'><b>Original:</b> {text}<br/>"
                               f"<b>Processed:</b> <span style='color:#1a1a2e'>{processed}</span></div>")
            pred_block = (f"<div style='border-left:4px solid {col};padding:8px 12px;background:#f9f9f9;"
                          f"border-radius:4px;margin:6px 0'><b>Prediction:</b> "
                          f"<span style='color:{col};font-size:15px'>{label.upper()}</span>"
                          f" &nbsp;|&nbsp; <b>Confidence:</b> {round(conf*100,1)}%{agreement_tag}</div>")
            bar_html = ""
            for lbl, prob in res.get("scores", {}).items():
                pct = round(prob * 100, 1)
                bar_html += (f"<div style='display:flex;align-items:center;margin:2px 0'>"
                             f"<span style='width:120px;font-size:12px'>{lbl}</span>"
                             f"<div style='background:{_colour(lbl)};width:{max(pct,1)}%;height:14px;"
                             f"border-radius:3px;margin-right:6px'></div>"
                             f"<span style='font-size:12px'>{pct}%</span></div>")
            with _output:
                clear_output()
                display(widgets.HTML(_steps_html(steps) + input_block + pred_block
                                     + f"<div style='margin:6px 0'><b>Score breakdown:</b><br/>{bar_html}</div>"))
                if "block_hash" in res:
                    print(f"\n✅ Recorded on-chain | block_hash={res['block_hash'][:20]}...")
                print(f"   Source: {src} | Model: {model}")
                if notes:
                    print(f"   Notes: {notes}")
        except Exception:
            with _output:
                clear_output(); print("❌ Error during inference:\n"); traceback.print_exc()

    def _on_verify(btn):
        with _output:
            clear_output(); print("🔗 Verifying all blockchains …")
            v = fl_system.registry.verify_all()
            print(f"  Overall Valid : {'✅ YES' if v['fully_valid'] else '❌ NO'}")
            print("  Global Chain  : " + ("✅ VALID" if v["global_chain"]["valid"] else "❌ INVALID")
                  + f" ({v['global_chain']['length']} blocks)")
            for cid, res in v["client_chains"].items():
                print(f"  {cid:<50} " + ("✅ VALID" if res["valid"] else "❌ INVALID")
                      + f" ({res['length']} blocks)")
            print("\n  Cross-Chain Merkle:")
            for cc in v["cross_chain_integrity"]:
                print(f"    Round {cc['fl_round']:02d}  GlobalBlock #{cc['global_block']:03d}  "
                      + ("✅ MATCH" if cc["merkle_match"] else "❌ MISMATCH"))
            print("\n✅ Verification complete.")

    def _on_audit(btn):
        with _output:
            clear_output(); print(fl_system.registry.audit_trail())

    def _on_export(btn):
        with _output:
            clear_output()
            path = fl_system.registry.export_all("blockchain_export.json")
            print(f"✅ Blockchain exported → {path}")
            fl_system.registry.save_audit_report("audit_report.txt")
            print("✅ Audit report   → audit_report.txt")

    _analyse_btn.on_click(_on_analyse); _verify_btn.on_click(_on_verify)
    _audit_btn.on_click(_on_audit);     _export_btn.on_click(_on_export)

    display(widgets.VBox([
        _title, widgets.Label("Input text:"), _text_input,
        widgets.HBox([_source_dd, _client_dd]),
        _translit_toggle, _unmask_toggle,
        widgets.Label("Analyst notes:"), _notes_input,
        widgets.HBox([_analyse_btn, _verify_btn, _audit_btn, _export_btn]),
        _output,
    ], layout=widgets.Layout(padding="12px", max_width="820px")))

else:
    # ── CLI fallback (uses ensemble_predict, which is now defined) ────────────
    print("\n--- Evidence Detection (CLI fallback) ---")
    for t in ["Ami tomake hate kori", "আমি ভালো আছি", "অ** প**ষ"]:
        processed, steps = _preprocess_text(t, do_translit=True, do_unmask=True)
        for lbl, before, after in steps:
            print(f"  [{lbl}] {before} → {after}")
        res = fl_system.ensemble_predict([processed])[0]
        print(f"  IN  : {t}")
        print(f"  PRED: {res['prediction']} ({round(res['confidence']*100,1)}%)\n")


In [ ]:
# ============================================================
# 3.13 · Final blockchain audit report + (optional) GPU memory report
# ============================================================
print(fl_system.registry.audit_trail())

def gpu_memory_report():
    """Print live CUDA tensors; no-op on CPU."""
    if not torch.cuda.is_available():
        print("No CUDA device — skipping GPU memory report.")
        return
    print(f"Allocated : {torch.cuda.memory_allocated() / 1024**3:.2f} GiB")
    print(f"Reserved  : {torch.cuda.memory_reserved()  / 1024**3:.2f} GiB")
    total = torch.cuda.get_device_properties(0).total_memory
    print(f"Free      : {(total - torch.cuda.memory_allocated()) / 1024**3:.2f} GiB\n")
    seen = set()
    for obj in gc.get_objects():
        try:
            if torch.is_tensor(obj) and obj.is_cuda and id(obj) not in seen:
                seen.add(id(obj))
                mb = obj.element_size() * obj.nelement() / 1024**2
                print(f"  {str(obj.dtype):<20} {str(tuple(obj.shape)):<30} {mb:.1f} MiB")
        except Exception:
            pass

gpu_memory_report()
print("\n✅ Pipeline complete. Artefacts: forensic_dataset_bn.csv, checkpoints/, "
      "blockchains/, fedavg_results.json, audit_report.txt, blockchain_export.json")
